<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-06/NB06_klinik_metin_madenciligi.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html)

# Week 06: NLP and clinical text mining

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lab.html) · [PDF notes](Week06_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-06/NB06_clinical_text_mining.ipynb)

</div>

**How to use this Jupyter notebook.** The notebook follows the practice stops of the lecture page. Every stop has three parts, and each part also runs on its own: Python basics, application with ready prompts, and the ready application. **Python basics** teaches the Python that the stop needs, with small examples from the topic of the week; its exercises check themselves. **Prompt and application** gives ready prompts for an artificial intelligence (AI) assistant: The assistant writes the code, you run it, and a check cell tells you whether the result is right. **Ready application** is the complete working Python program of the stop, with notes that explain it.

Go through the three parts in this order. If you already know Python, you may start with the prompts; if you only want to see the result, open the ready application. The weekly project at the end stands on its own as well.

**Start cells.** Every part opens with a start cell. It prepares the course package and the packages of the week, and it repeats the data calls and the code of the earlier parts that the cells of the part use. The part therefore runs without errors even when you start there. If you have run the earlier parts in this session, you do not need to run the start cell; running it again does no harm. A few sections of the ready applications, which the lecture page opens directly, have a start cell of their own.

**Data.** The reports of the week are synthetic: The virtual clinic of the course package writes the discharge summaries and the angiography reports of both hospitals in the Jupyter notebook, in a few seconds. The first time a cell loads the tiny Turkish language model of the course, ytu-ce-cosmos/turkish-tiny-bert-uncased, the package Transformers downloads it from the Hugging Face Hub, about 18.5 MB, and later runs use the copy on the computer. The model is published under the MIT license. The ready application of Stop 2 also downloads the tokenizer of BioBERT, a few hundred kilobytes.

**Finding your place.** The table of contents lists every stop and part. In Colab it opens with the list icon on the left; in Jupyter, with View → Table of Contents. Every part ends with a link back to the lecture page. In Colab, your changes are kept only in a copy saved to Drive: Use File → Save a copy in Drive.

**Where it runs.** The notebook runs in Jupyter (Anaconda) on your own computer and in Google Colab. Every data set here is synthetic or comes from a public research database. Real patient data belong only on a computer of your institution, under its rules, and never in a cloud notebook or in an AI chat.

## 0. Setup

**First use of Jupyter Notebook.** A notebook is made of cells. This cell holds text; the grey cells hold Python code. Click into the code cell below and press Shift+Enter: The cell runs, its result appears below it, and the number in the brackets on its left, such as `[1]`, shows the order of the run. The kernel, the Python process behind the notebook, remembers everything that the cells have run, so the cells run from top to bottom. **Kernel → Restart Kernel and Run All Cells** repeats the whole notebook, and Ctrl+S (Cmd+S on a Mac) saves it as an `.ipynb` file.

**The start cell.** The code cell below prepares the session. It finds the course package `vclinic` in the course folder; if the notebook was opened on its own, for example in Google Colab, it downloads the package from GitHub once. It then loads the packages, chooses the language of the messages, installs the package of the week, `transformers`, with pip when it is missing and prepares the expected answers of the exercises. Its last lines import the libraries of the week. Every part of the notebook opens with a start cell that holds the same setup, together with the data calls of the part.

In [ ]:
# Start cell of the Jupyter notebook
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the libraries of the week

import difflib
import re
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

## Stop 1: Six diagnoses from discharge summaries: rules and a model

*Lecture page: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#stop-1)*

The six chronic diagnoses are read from the discharge summaries of the virtual clinic, first by searching for their words, then with the assertion of every mention, then with the drugs and two laboratory values, and finally by a model that learns from the training summaries. Every method is judged on the test set of hospital A, and the model also on the summaries of hospital B.

**At the end of this stop:** The positive predictive value of the keyword rule for atrial fibrillation (0.19), the macro F1 with the assertion of every mention (0.856) and with the drugs and the laboratory values (0.98), and the macro F1 of the TF-IDF model on hospital B (0.708).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 1 · Python basics

This week the data are text. In Python, a text is a *string*: a sequence of characters between quotes. This part shows how to look into a string, how to change it with its methods, how to bring Turkish text to small letters correctly, how to find patterns with regular expressions, how to cut a text into tokens and, finally, how to call a pretrained language model from the Hugging Face Hub.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 1 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

In [ ]:
# Python practice: a small example for learning the language
text = "Öykü: 83 yaşında kadın hasta, eforla gelen nefes darlığı yakınmasıyla kardiyoloji polikliniğine başvurdu. Özgeçmiş: Bilinen kronik hastalığı yok. Atriyal fibrilasyon tanımlamıyor."
print(len(text), "characters")
print(text[:5], "|", text[-31:])
print(text.lower()[:40])
print("tanımlamıyor" in text, "diyabet" in text)
print(text.split(". ")[2])
print(f"The text has {text.count('a')} letters a.")

**Python note.** A string is written between quotes, and `len` counts its characters. Square brackets take a part of it: `text[:5]` is the first five characters and `text[-31:]` the last 31, since negative positions count from the end. A *method* is a function that belongs to a value and is called with a dot: `text.lower()` gives a copy in small letters, `text.split(". ")` cuts the text at every full stop followed by a space and gives a list, and `text.count("a")` counts a letter. `"tanımlamıyor" in text` asks whether one string occurs in another and gives `True` or `False`. An f-string, `f"... {value} ..."`, puts values into a text.

**On the lecture page:** [Worked example: Two lower cases of "İNFERİOR"](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-two-lower-cases-of-i-nferi-or). The cell below compares the two lower cases of the same word.

In [ ]:
# Python practice: two letters i
word = "İNFERİOR"
python_lower = word.lower()


def tr_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


print(python_lower, len(python_lower), "|", tr_lower(word), len(tr_lower(word)))
print([hex(ord(c)) for c in python_lower[:3]])
print(tr_lower("PROKSİMAL KISIMDA"), "|", "PROKSİMAL KISIMDA".lower())

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c102)*

**Python note.** `def tr_lower(text):` defines a function of one line that the whole week uses: It replaces I by ı and İ by i before it calls `lower()`, so that Turkish words keep their letters and their length. Python's own `lower()` turns İ into two characters: `ord` gives the number of a character in Unicode, and `hex` writes it in base 16. The second character, 0x307, is a dot above, which makes the word longer by one and different from "inferior". And `lower()` turns KISIMDA into "kisimda", with two letters i that Turkish does not have.

In [ ]:
# Python practice: regular expressions
lab = "Laboratuvar: kreatinin 0,95 mg/dL, eGFR 59 mL/dk/1,73 m², HbA1c %7,8, LDL 83 mg/dL."
m = re.search(r"eGFR (\d+)", lab)
print(m.group(0), "→", int(m.group(1)))
print(re.findall(r"\d+(?:,\d+)?", lab))
hba1c = re.search(r"HbA1c %(\d+,\d+)", lab).group(1)
print(float(hba1c.replace(",", ".")))
for sentence in ["Diyabet yok.", "Prediyabet nedeniyle diyet önerildi."]:
    print(sentence, "| in:", "diyabet" in tr_lower(sentence), "| whole word:", bool(re.search(r"(?<!\w)diyabet(?!\w)", tr_lower(sentence))))

**Python note.** A *regular expression* is a small pattern for text, written as a raw string `r"..."` so that the backslashes stay as they are. `\d` is any digit and `+` means one or more, so `\d+` is a whole number; the parentheses `( )` mark a part to keep, which `m.group(1)` returns. `re.search` finds the first match and `re.findall` all of them. `(?:,\d+)?` adds an optional decimal part with a comma, the Turkish way, and `.replace(",", ".")` turns it into a number that `float` can read. `(?<!\w)` and `(?!\w)` say "no letter before" and "no letter after": a whole word. With them, "diyabet" is no longer found inside "prediyabet".

**On the lecture page:** [Worked example: WordPiece cuts "yetmezliğinin"](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-wordpiece-cuts-yetmezli-inin). The cell below cuts a sentence with the same word into the tokens of the model of the course.

In [ ]:
# Python practice: tokens
sentence = "Dekompanse kalp yetmezliğinin nedeni araştırıldı."
print(sentence.split())
print(re.findall(r"\w+", tr_lower(sentence)))
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
encoded = tokenizer(tr_lower(sentence))
print(encoded["input_ids"])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print(len(tokenizer), "pieces in the vocabulary")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c106)*

**Python note.** `split()` cuts at spaces and keeps the punctuation on the words; `re.findall(r"\w+", ...)` keeps only runs of letters and digits. A language model uses its own tokenizer. `AutoTokenizer.from_pretrained(MODEL)` downloads the files of the tokenizer of the model from the Hugging Face Hub the first time and loads them; the name of the model is that of its page on the Hub. Called on a text, the tokenizer gives the numbers of the tokens, `input_ids`, with the special tokens [CLS] at the start and [SEP] at the end; `convert_ids_to_tokens` turns the numbers back into pieces. "yetmezliğinin" becomes "yetmezliği" and "##nin".

In [ ]:
# Python practice: calling a pretrained model
fill = pipeline("fill-mask", model=MODEL)
for guess in fill("hastanın kan [MASK] yüksekti."):
    print(f"{guess['token_str']:12} {guess['score']:.3f}")

**Python note.** `pipeline("fill-mask", model=MODEL)` downloads the weights of the model, about 18.5 MB, and wraps the tokenizer and the model into one function. Given a sentence with the special token [MASK], it returns the five most probable tokens for the gap with their probabilities, each as a small dictionary; `guess['token_str']` and `guess['score']` read its keys. The first guess, "basıncı", completes "kan basıncı", the blood pressure: The model learned this from general Turkish text, not from the reports of the clinic.

**Quick check.** What does `re.findall(r"\d+", "EF %55, eGFR 59")` return?

<details><summary>Answer</summary>

The list `['55', '59']`: every run of digits, as strings. They become numbers only with `int` or `float`.

</details>

**Exercise 1.1.** How many words, runs of letters and digits after the Turkish lower case, does the discharge summary of the case, patient VC-2444, have? The summary is `reports.archive()` in the row of the patient, column `text_tr`.

In [ ]:
data = exercise_table("1.1")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
case_text = reports.archive().set_index("patient_id").loc["VC-2444", "text_tr"]
answer = len(re.findall(r"\w+", tr_lower(case_text)))
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** Read the eGFR of the case from its summary with a regular expression and keep it as a whole number.

In [ ]:
data = exercise_table("1.2")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
case_text = reports.archive().set_index("patient_id").loc["VC-2444", "text_tr"]
answer = int(re.search(r"eGFR (\d+)", case_text).group(1))
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** How many tokens does the tokenizer of the course give for the summary of the case after the Turkish lower case, without the two special tokens?

In [ ]:
data = exercise_table("1.3")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
case_text = reports.archive().set_index("patient_id").loc["VC-2444", "text_tr"]
answer = len(tokenizer(tr_lower(case_text))["input_ids"]) - 2
teach.check("Exercise 1.3", answer)

**«Stop 1 · Python basics» is complete.** Next comes «Stop 1 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#last-two-hours-advanced-clinical-nlp-applications) and continue reading at «Last two hours: Advanced clinical NLP applications».

### Stop 1 · Prompt and application

This part has four prompts: «A keyword rule», «The assertion of every mention», «Drugs and laboratory values» and «A model that learns from the summaries: TF-IDF and logistic regression».

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Working with prompts.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab and opens with the Gemini button at the top of the window; ChatGPT, Claude, Copilot or Gemini in another browser tab work as well. All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-06/PROMPTS.md), where GitHub shows a copy button next to each one.

**Two names in every prompt: "data" and "answer".** An AI assistant usually does not see the notebook, so every prompt describes the data in words. For each prompt, the data cell puts its data into a table named "data", and the prompt describes the rows and the columns of this table. The prompt also asks the assistant to keep the result under the name "answer", and the check cell reads the result under this name. These are the only two names from the code that appear in the prompts of the course.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

**Start cell.** For «Stop 1 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

from vclinic import reports
import re

if "archive" not in globals():
    archive = reports.archive()
    LABELS = list(reports.LABELS)
    texts = archive["text_tr"].to_numpy()
    y = archive[LABELS].to_numpy()
    split = archive["split"].to_numpy()
    tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


    def tr_lower(text):
        """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """The words of a text in lower case: runs of letters and digits."""
        return re.findall(r"\w+", tr_lower(text))
_COLS = ["HF", "AF", "CAD", "HT", "DM", "CKD"]
_SETS = {"training": "training", "validation": "validation", "test": "test", "external": "external"}


def _summaries(mask, with_set=False, with_entities=False):
    """The summaries of a set with the six diagnoses of the registry, as the prompts describe them."""
    t = pd.DataFrame({"summary": texts[mask]})
    if with_set:
        t.insert(0, "set", [_SETS[s] for s in split[mask]])
    if with_entities:
        t["entities"] = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    for k, c in enumerate(_COLS):
        t[c] = y[mask, k]
    return t.reset_index(drop=True)

print(len(archive), "summaries;", {_SETS[s]: int((split == s).sum()) for s in _SETS})

#### Prompt 1.1: A keyword rule

**Goal.** Search the summaries for the words of six diagnoses and see where a search for words goes wrong.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well a simple search for the words of a diagnosis finds six chronic diagnoses in discharge summaries.  
> Data: The notebook holds a table named "data" with the 600 discharge summaries of the test set of hospital A of the virtual clinic, a synthetic registry. Each row is a patient: "summary" holds the discharge summary in Turkish, and the six columns "HF", "AF", "CAD", "HT", "DM" and "CKD" hold 0 or 1, with 1 when the registry records heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease, and 0 otherwise.  
> Task: Bring every text to small letters by the Turkish rules: First turn I into ı and İ into i, then make the other letters small. Look for these words of each diagnosis as whole words, so that no word is found inside a longer word: heart failure "kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky" and "kky"; atrial fibrillation "atriyal fibrilasyon", "af" and "afib"; coronary artery disease "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" and "akut koroner sendrom"; hypertension "hipertansiyon" and "ht"; diabetes mellitus "diyabet", "diyabetes mellitus" and "dm"; chronic kidney disease "kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh" and "kby". A diagnosis is present when at least one of its words is found anywhere in the summary. Compare the six decisions with the six columns of the registry and compute, for each diagnosis, the sensitivity, the positive predictive value and the F1 score (twice the true positives divided by twice the true positives plus the false positives plus the false negatives), and the mean of the six F1 scores, the macro F1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with one row for each diagnosis and the three values, and the macro F1. Keep the positive predictive value of atrial fibrillation, with three decimals, under the name "answer".  
> Test: Every value must lie between 0 and 1, and the summary of patient VC-2444 must be called positive for heart failure, atrial fibrillation and coronary artery disease.  

In [ ]:
# Data of Prompt 1.1: the table "data" that the prompt describes
data = _summaries(te)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.1
teach.check_step("Prompt 1.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.1 (run it only if the code of the assistant does not pass the check)
data = _summaries(te)
terms = {"HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def score(decisions):
    rows = []
    for c in terms:
        y_true = data[c].to_numpy() == 1
        y_pred = np.array(decisions[c]) == 1
        tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
        rows.append({"diagnosis": c, "sensitivity": round(tp / max(1, tp + fn), 3),
                         "PPV": round(tp / max(1, tp + fp), 3), "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
    table = pd.DataFrame(rows)
    print(table)
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table

decisions = {c: [] for c in terms}
for text in data["summary"]:
    low = turkish_lower(text)
    for c, words in terms.items():
        found = any(re.search(r"(?<!\w)" + re.escape(w) + r"(?!\w)", low) for w in words)
        decisions[c].append(int(found))
table = score(decisions)
answer = float(table.loc[table["diagnosis"] == "AF", "PPV"].iloc[0])
teach.check_step("Prompt 1.1", "answer", globals())

**What the result shows.** The positive predictive value for atrial fibrillation is 0.19: Four of five alarms are false, because the rule counts "Atriyal fibrilasyon izlenmedi" and "AF tanımlamıyor" as atrial fibrillation. Chronic kidney disease has the lowest sensitivity, because most summaries never name it.

#### Prompt 1.2: The assertion of every mention

**Goal.** Leave out the mentions that are negated, possible or in a relative.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how much a search for the words of six diagnoses in discharge summaries improves when mentions that are negated, only suspected or about a relative are left out.  
> Data: The notebook holds a table named "data" with the 600 discharge summaries of the test set of hospital A of the virtual clinic, a synthetic registry. Each row is a patient: "summary" holds the discharge summary in Turkish, and the six columns "HF", "AF", "CAD", "HT", "DM" and "CKD" hold 0 or 1, with 1 when the registry records heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease, and 0 otherwise.  
> Task: Bring every text to small letters by the Turkish rules: First turn I into ı and İ into i, then make the other letters small. Look for these words of each diagnosis as whole words, so that no word is found inside a longer word: heart failure "kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky" and "kky"; atrial fibrillation "atriyal fibrilasyon", "af" and "afib"; coronary artery disease "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" and "akut koroner sendrom"; hypertension "hipertansiyon" and "ht"; diabetes mellitus "diyabet", "diyabetes mellitus" and "dm"; chronic kidney disease "kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh" and "kby". A mention does not count when, after it and before the end of its sentence or the next mention of another diagnosis, one of these words follows: "yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı" or "(-)" (negated), or "şüphe", "olası" or "ön tanı" (possible). It does not count either when, earlier in the same sentence, one of these words stands: "babası", "annesi", "kardeşi", "babada", "annede" or "soygeçmiş" (a relative). A diagnosis is present when at least one of its mentions counts. Compare the six decisions with the six columns of the registry and compute, for each diagnosis, the sensitivity, the positive predictive value and the F1 score (twice the true positives divided by twice the true positives plus the false positives plus the false negatives), and the mean of the six F1 scores, the macro F1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with one row for each diagnosis and the three values, and keep the macro F1, with three decimals, under the name "answer".  
> Test: The positive predictive value of atrial fibrillation must be higher than with Prompt 1.1, and the summary of patient VC-2444 must be positive only for heart failure.  

In [ ]:
# Data of Prompt 1.2: the table "data" that the prompt describes
data = _summaries(te)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.2
teach.check_step("Prompt 1.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.2 (run it only if the code of the assistant does not pass the check)
data = _summaries(te)
terms = {"HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def score(decisions):
    rows = []
    for c in terms:
        y_true = data[c].to_numpy() == 1
        y_pred = np.array(decisions[c]) == 1
        tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
        rows.append({"diagnosis": c, "sensitivity": round(tp / max(1, tp + fn), 3),
                         "PPV": round(tp / max(1, tp + fp), 3), "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
    table = pd.DataFrame(rows)
    print(table)
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table

negation = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
possible = ["şüphe", "olası", "ön tanı"]
relative = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]


def present(low):
    mentions = sorted((m.start(), m.end(), c) for c, words in terms.items() for w in words
                      for m in re.finditer(r"(?<!\w)" + re.escape(w) + r"(?!\w)", low))
    found = set()
    for i, (start, end, c) in enumerate(mentions):
        next_other = next((m[0] for m in mentions[i + 1:] if m[2] != c), len(low))
        stops = [x for x in (low.find(".", end), low.find(";", end), next_other, len(low)) if x >= end]
        after = low[end:min(stops)]
        sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
        before = low[sentence_start:start]
        if any(w in before for w in relative):
            continue
        if any(w in after for w in negation + possible):
            continue
        found.add(c)
    return found


decisions = {c: [] for c in terms}
for text in data["summary"]:
    found = present(turkish_lower(text))
    for c in terms:
        decisions[c].append(int(c in found))
table = score(decisions)
answer = round(float(table["F1"].mean()), 3)
teach.check_step("Prompt 1.2", "answer", globals())

**What the result shows.** The macro F1 rises from 0.666 with the keyword rule to 0.856. The positive predictive values reach 1, and the remaining errors are missed diagnoses: diagnoses that the summary never names.

#### Prompt 1.3: Drugs and laboratory values

**Goal.** Add the evidence that is not a word for the disease: drugs, eGFR and HbA1c.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to find six chronic diagnoses in discharge summaries also when the summary does not name them but lists a drug that is given for them or a laboratory value that defines them.  
> Data: The notebook holds a table named "data" with the 600 discharge summaries of the test set of hospital A of the virtual clinic, a synthetic registry. Each row is a patient: "summary" holds the discharge summary in Turkish, and the six columns "HF", "AF", "CAD", "HT", "DM" and "CKD" hold 0 or 1, with 1 when the registry records heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease, and 0 otherwise.  
> Task: Bring every text to small letters by the Turkish rules: First turn I into ı and İ into i, then make the other letters small. Look for these words of each diagnosis as whole words, so that no word is found inside a longer word: heart failure "kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky" and "kky"; atrial fibrillation "atriyal fibrilasyon", "af" and "afib"; coronary artery disease "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" and "akut koroner sendrom"; hypertension "hipertansiyon" and "ht"; diabetes mellitus "diyabet", "diyabetes mellitus" and "dm"; chronic kidney disease "kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh" and "kby". A mention does not count when, after it and before the end of its sentence or the next mention of another diagnosis, one of these words follows: "yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı" or "(-)" (negated), or "şüphe", "olası" or "ön tanı" (possible). It does not count either when, earlier in the same sentence, one of these words stands: "babası", "annesi", "kardeşi", "babada", "annede" or "soygeçmiş" (a relative). A diagnosis is present when at least one of its mentions counts. A diagnosis is also present when the summary names one of its drugs, as a whole word or the start of a word: heart failure "furosemid" or "sakubitril"; atrial fibrillation "apiksaban", "rivaroksaban", "edoksaban" or "dabigatran"; coronary artery disease "klopidogrel", "tikagrelor", "pkg", "kabg" or "stent"; hypertension "amlodipin" or "hidroklorotiyazid"; diabetes mellitus "metformin", "gliklazid", "insülin" or "sitagliptin". Chronic kidney disease is present when the first number after "egfr" is below 60, and diabetes mellitus when the first number after "hba1c" and an optional percent sign is 6.5 or more; these numbers use a decimal comma. Compare the six decisions with the six columns of the registry and compute, for each diagnosis, the sensitivity, the positive predictive value and the F1 score (twice the true positives divided by twice the true positives plus the false positives plus the false negatives), and the mean of the six F1 scores, the macro F1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with one row for each diagnosis and the three values, and keep the macro F1, with three decimals, under the name "answer".  
> Test: The sensitivity of chronic kidney disease must be 1, and the summary of patient VC-2444 must be positive for heart failure, hypertension and chronic kidney disease.  

In [ ]:
# Data of Prompt 1.3: the table "data" that the prompt describes
data = _summaries(te)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.3
teach.check_step("Prompt 1.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.3 (run it only if the code of the assistant does not pass the check)
data = _summaries(te)
terms = {"HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def score(decisions):
    rows = []
    for c in terms:
        y_true = data[c].to_numpy() == 1
        y_pred = np.array(decisions[c]) == 1
        tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
        rows.append({"diagnosis": c, "sensitivity": round(tp / max(1, tp + fn), 3),
                         "PPV": round(tp / max(1, tp + fp), 3), "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
    table = pd.DataFrame(rows)
    print(table)
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table

negation = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
possible = ["şüphe", "olası", "ön tanı"]
relative = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
drugs = {"HF": ["furosemid", "sakubitril"], "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
           "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"], "HT": ["amlodipin", "hidroklorotiyazid"],
           "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"]}


def present(low):
    mentions = sorted((m.start(), m.end(), c) for c, words in terms.items() for w in words
                      for m in re.finditer(r"(?<!\w)" + re.escape(w) + r"(?!\w)", low))
    found = set()
    for i, (start, end, c) in enumerate(mentions):
        next_other = next((m[0] for m in mentions[i + 1:] if m[2] != c), len(low))
        stops = [x for x in (low.find(".", end), low.find(";", end), next_other, len(low)) if x >= end]
        after = low[end:min(stops)]
        sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
        if any(w in low[sentence_start:start] for w in relative):
            continue
        if any(w in after for w in negation + possible):
            continue
        found.add(c)
    for c, words in drugs.items():
        if any(re.search(r"(?<!\w)" + re.escape(w), low) for w in words):
            found.add(c)
    egfr = re.search(r"egfr\s*%?\s*(\d+(?:,\d+)?)", low)
    if egfr and float(egfr.group(1).replace(",", ".")) < 60:
        found.add("CKD")
    hba1c = re.search(r"hba1c\s*%?\s*(\d+(?:,\d+)?)", low)
    if hba1c and float(hba1c.group(1).replace(",", ".")) >= 6.5:
        found.add("DM")
    return found


decisions = {c: [] for c in terms}
for text in data["summary"]:
    found = present(turkish_lower(text))
    for c in terms:
        decisions[c].append(int(c in found))
table = score(decisions)
answer = round(float(table["F1"].mean()), 3)
teach.check_step("Prompt 1.3", "answer", globals())

**What the result shows.** The macro F1 reaches 0.98. Diabetes and chronic kidney disease are now found in every patient. Hypertension stays the weakest diagnosis: "esansiyel hipertansiyon" and "hipertansyon" are missing from the list of words, and the drugs of the summaries that name neither, ACE inhibitors and angiotensin receptor blockers, are given for heart failure as well.

#### Prompt 1.4: A model that learns from the summaries: TF-IDF and logistic regression

**Goal.** Train a model on the summaries of hospital A and judge it on hospital A and on hospital B.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to train a model that finds six chronic diagnoses in discharge summaries from examples, and to know how well it works in its own hospital and in another hospital that writes in another style.  
> Data: The notebook holds a table named "data" with the 3,600 discharge summaries of the virtual clinic, a synthetic registry. Each row is a patient: "set" gives the set of the summary ("training", 1,800 summaries; "validation", 600; "test", 600, all three from hospital A; "external", 600 summaries from hospital B, written in a short style with (+) and (-)), "summary" holds the discharge summary in Turkish, and the six columns "HF", "AF", "CAD", "HT", "DM" and "CKD" hold 0 or 1, with 1 when the registry records heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease, and 0 otherwise.  
> Task: Bring every text to small letters by the Turkish rules: First turn I into ı and İ into i, then make the other letters small. Turn every summary into a vector of TF-IDF weights of its words and of its pairs of neighbouring words; a word is a run of letters and digits, and only the terms that occur in at least two training summaries are kept. Learn the terms and their weights from the training summaries only, use the logarithm of the count plus one as the frequency of a term, and scale every vector to length 1. For each diagnosis, train a logistic regression with an inverse regularisation strength of 10 on the training summaries. On the validation summaries, choose for each diagnosis the threshold of the probability, from 0.05 to 0.95 in steps of 0.05, where its F1 score is highest. With these thresholds, compute the F1 score of each diagnosis and the macro F1 on the test summaries and on the external summaries.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with the F1 score of each diagnosis on the test and on the external summaries, and keep the macro F1 on the external summaries, with three decimals, under the name "answer".  
> Test: The terms must come from the training summaries only, the thresholds from the validation summaries only, and the macro F1 on the test summaries must be above 0.9.  

In [ ]:
# Data of Prompt 1.4: the table "data" that the prompt describes
data = _summaries(np.ones(len(texts), dtype=bool), with_set=True)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.4
teach.check_step("Prompt 1.4", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.4 (run it only if the code of the assistant does not pass the check)
data = _summaries(np.ones(len(texts), dtype=bool), with_set=True)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

cols = ["HF", "AF", "CAD", "HT", "DM", "CKD"]
sets = {s: data[data["set"] == s] for s in ("training", "validation", "test", "external")}
vectorizer = TfidfVectorizer(preprocessor=lambda t: t.replace("I", "ı").replace("İ", "i").lower(), token_pattern=r"(?u)\b\w+\b",
                             ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = {"training": vectorizer.fit_transform(sets["training"]["summary"])}
for s in ("validation", "test", "external"):
    X[s] = vectorizer.transform(sets[s]["summary"])


def f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true) == 1, np.asarray(y_pred)
    tp = int((y_pred & y_true).sum())
    return 2 * tp / max(1, 2 * tp + int((y_pred & ~y_true).sum()) + int((~y_pred & y_true).sum()))


grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows = []
for c in cols:
    model = LogisticRegression(C=10, max_iter=2000).fit(X["training"], sets["training"][c])
    p_val = model.predict_proba(X["validation"])[:, 1]
    threshold = float(grid[int(np.argmax([f1(sets["validation"][c], p_val >= t) for t in grid]))])
    row = {"diagnosis": c, "threshold": threshold}
    for s in ("test", "external"):
        row[f"F1 {s}"] = round(f1(sets[s][c], model.predict_proba(X[s])[:, 1] >= threshold), 3)
    rows.append(row)
table = pd.DataFrame(rows)
print(table)
print("Macro F1, test:", round(float(table["F1 test"].mean()), 3))
answer = round(float(table["F1 external"].mean()), 3)
print("Macro F1, external:", answer)
teach.check_step("Prompt 1.4", "answer", globals())

**What the result shows.** The model is almost perfect on the test set of hospital A and reaches only 0.708 on hospital B. Its heaviest terms come from the way hospital A writes, a phrase of its ECG sentence or a dose, and the short style of hospital B, with "AF (-)", gives it words that it took for diseases.

**«Stop 1 · Prompt and application» is complete.** Next comes «Stop 1 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#last-two-hours-advanced-clinical-nlp-applications) and continue reading at «Last two hours: Advanced clinical NLP applications».

### Stop 1 · Ready application

This part reads the report archive of the virtual clinic, builds a keyword rule for six diagnoses and an assertion rule that also uses drugs and laboratory values, trains a TF-IDF model with logistic regression on the summaries of hospital A and judges everything on hospital B. Lemmatisation and word vectors learned from the archive close the part.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 1 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

#### The report archive of the virtual clinic

*Lecture page: [Text as data: The report archive of the virtual clinic](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#text-as-data-the-report-archive-of-the-virtual-clinic)*

The virtual clinic writes one discharge summary for each of the 3,600 patients of a registry: 3,000 in full sentences by hospital A, in Turkish and in English, and 600 in a short style by hospital B. The registry knows the six diagnoses of every patient, so every method of the week can be checked against it.

**From the formulas to the code.** The table has one row per patient; `text_tr` is the summary in Turkish, and the six columns `HF`, `AF`, `CAD`, `HT`, `DM` and `CKD` (heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease) are 1 or 0. The column `split` gives the set: `training`, `validation`, `test` or `external`. The prevalence of a diagnosis in a set is the mean of its column: `archive[archive["split"] == "test"]["AF"].mean()`.

In [ ]:
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


def tr_lower(text):
    """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
    return text.replace("I", "ı").replace("İ", "i").lower()


def words(text):
    """The words of a text in lower case: runs of letters and digits."""
    return re.findall(r"\w+", tr_lower(text))

print(archive.shape)
print(archive.groupby("split")[LABELS].sum())
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
print()
print(texts[case])

**Python note.** `reports.archive()` writes the archive in a few seconds and returns it as a table. `groupby("split")[LABELS].sum()` counts the patients with each diagnosis in each set. `texts`, `y` and `split` are the columns as arrays, and the four masks `tr`, `va`, `te` and `ex` pick the rows of the training, validation, test and external sets, as in the earlier weeks. The function `words` of the code cell returns the words of a text after `tr_lower`.

In [ ]:
n_words = np.array([len(words(t)) for t in texts])
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist(n_words[tr | va | te], bins=40, alpha=0.7, label="hospital A")
ax.hist(n_words[ex], bins=40, alpha=0.7, label="hospital B")
ax.set_xlabel("words per summary")
ax.set_ylabel("summaries")
ax.legend()
plt.show()
print("Hospital B, first summary:", texts[ex][0])

**Observe and reflect.** Read the summary of the case and one summary of hospital B. Which of the six diagnoses can you find in each, and from which words? Which of them would a search for the names of the diseases miss?

**Exercise A.** How many patients of the test set of hospital A have at least three of the six diagnoses in the registry?

In [ ]:
data = exercise_table("A")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((y[te].sum(axis=1) >= 3).sum())
teach.check("Exercise A", answer)

#### A keyword rule and its traps

*Lecture page: [A rule first: Searching for words and its traps](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#a-rule-first-searching-for-words-and-its-traps)*

The first rule calls a diagnosis present when one of its words occurs anywhere in the summary, as a whole word after the Turkish lower case. It is judged per diagnosis with the sensitivity, the positive predictive value and F1, and over all six with the macro F1.

**From the formulas to the code.** $F_1 = 2\,TP/(2\,TP + FP + FN)$: the third value of `se_ppv_f1`. The macro F1 is the mean of the column `F1` of `label_table`. The whole word is the pattern `(?<!\w)(...)(?!\w)` of `PATTERNS`.

**Start cell.** For the section «A keyword rule and its traps», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «A keyword rule and its traps»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
    return text.replace("I", "ı").replace("İ", "i").lower()
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])

**On the lecture page:** [Worked example: F1 of the keyword rule for atrial fibrillation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-f1-of-the-keyword-rule-for-atrial-fibrillation). The cell below computes the sensitivity, the positive predictive value and the F1 of the keyword rule for the six diagnoses.

In [ ]:
def se_ppv_f1(y_true, y_pred):
    """Sensitivity, positive predictive value and F1 of a yes-or-no decision."""
    y_true, y_pred = np.asarray(y_true, dtype=bool), np.asarray(y_pred, dtype=bool)
    tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
    se = tp / max(1, tp + fn)
    ppv = tp / max(1, tp + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    return se, ppv, f1


def label_table(y_true, y_pred):
    """One row per diagnosis: prevalence, sensitivity, PPV and F1; the last row is the mean F1 (macro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["diagnosis", "with the diagnosis", "sensitivity", "PPV", "F1"])
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table


TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}


def find_terms(text):
    """Every mention of a diagnosis: (start, end, diagnosis), in the order of the text."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)


def keyword_labels(text):
    """A diagnosis counts when one of its words appears anywhere in the report."""
    present = {label for _, _, label in find_terms(text)}
    return [int(label in present) for label in LABELS]

keyword_test = np.array([keyword_labels(t) for t in texts[te]])
print(label_table(y[te], keyword_test))
print()
print("Case VC-2444, keyword rule:", dict(zip(LABELS, keyword_labels(texts[case]))))
print("Case VC-2444, registry:     ", dict(zip(LABELS, y[case])))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c133)*

**Python note.** `TERMS` is a dictionary from a diagnosis to its list of words. For every diagnosis, `PATTERNS` joins its words into one regular expression with `|`, which means "or"; the longer words come first, so that "kalp yetmezliği" wins over a shorter word inside it. `find_terms` returns every match as (start, end, diagnosis), and `keyword_labels` turns the set of diagnoses found into a list of six numbers 0 or 1. `label_table` compares two arrays of six columns and builds the table with pandas.

In [ ]:
# Where the rule calls atrial fibrillation without it
k = LABELS.index("AF")
false_af = np.where(te)[0][(keyword_test[:, k] == 1) & (y[te, k] == 0)]
print(len(false_af), "false alarms for AF; three sentences:")
for i in false_af[:3]:
    start, end, _ = next(m for m in find_terms(texts[i]) if m[2] == "AF")
    print("  ...", texts[i][max(0, start - 20):end + 30], "...")

**Observe and reflect.** The three sentences show the trap of negation. Look at the false alarms for coronary artery disease as well (change "AF" to "CAD"). Which other trap appears there?

**Exercise B.** What is the positive predictive value of the keyword rule for diabetes mellitus (`DM`) on the test set of hospital A? Keep it with three decimals.

In [ ]:
data = exercise_table("B")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
k = LABELS.index("DM")
answer = round(se_ppv_f1(y[te, k], keyword_test[:, k])[1], 3)
teach.check("Exercise B", answer)

#### Assertion, drugs and laboratory values

*Lecture page: [Negation, family history and uncertainty: The assertion of a mention](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#negation-family-history-and-uncertainty-the-assertion-of-a-mention)*

Every mention of a diagnosis gets an assertion. A trigger of negation or uncertainty after the mention, up to the end of the sentence or the next mention of another diagnosis, makes it negated or possible; a relative earlier in the sentence gives it to the family. Drugs and two laboratory values then add the diagnoses that the text does not name.

**From the formulas to the code.** The scope is the text between the end of the mention and the first of four positions: the next full stop, the next semicolon, `next_other(...)` and the end of the text, in `assertion`. A window of $w$ characters replaces it by `end + window`. The thresholds of the laboratory are `egfr < 60` and `hba1c >= 6.5` in `rule_labels`.

**Start cell.** For the section «Assertion, drugs and laboratory values», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Assertion, drugs and laboratory values»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
    return text.replace("I", "ı").replace("İ", "i").lower()
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
def se_ppv_f1(y_true, y_pred):
    """Sensitivity, positive predictive value and F1 of a yes-or-no decision."""
    y_true, y_pred = np.asarray(y_true, dtype=bool), np.asarray(y_pred, dtype=bool)
    tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
    se = tp / max(1, tp + fn)
    ppv = tp / max(1, tp + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    return se, ppv, f1
def label_table(y_true, y_pred):
    """One row per diagnosis: prevalence, sensitivity, PPV and F1; the last row is the mean F1 (macro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["diagnosis", "with the diagnosis", "sensitivity", "PPV", "F1"])
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Every mention of a diagnosis: (start, end, diagnosis), in the order of the text."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)

**On the lecture page:** [Worked example: A window and a scope in "KY (-) HT (+) DM (-)"](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-a-window-and-a-scope-in-ky-ht-dm). The cell below decides the assertion of every mention with the same scope.

In [ ]:
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]


def assertion(text, start, end, next_start, window=None):
    """The assertion of one mention: present, negated, possible or family.

    The scope after the mention ends at the end of the sentence or where the next mention of another diagnosis starts (next_start); with a window, it ends after that many characters instead."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"


def next_other(mentions, i, text):
    """Where the next mention of another diagnosis starts; the end of the text when there is none."""
    return next((m[0] for m in mentions[i + 1:] if m[2] != mentions[i][2]), len(text))


def assertion_labels(text, window=None):
    """A diagnosis counts when at least one of its mentions is present."""
    mentions = find_terms(text)
    present = set()
    for i, (start, end, label) in enumerate(mentions):
        if assertion(text, start, end, next_other(mentions, i, text), window) == "present":
            present.add(label)
    return [int(label in present) for label in LABELS]

mentions = find_terms(texts[case])
for i, (start, end, label) in enumerate(mentions):
    print(f"{label:4} {texts[case][start:end]:28} → {assertion(texts[case], start, end, next_other(mentions, i, texts[case]))}")
b = np.where(ex)[0][1]
print()
print(texts[b][:60])
for i, (start, end, label) in enumerate(find_terms(texts[b])):
    ms = find_terms(texts[b])
    print(f"{label:4} scope: {assertion(texts[b], start, end, next_other(ms, i, texts[b])):8} window of 40: {assertion(texts[b], start, end, 0, window=40)}")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c143)*

**Python note.** `assertion` works on the Turkish lower case of the text. `low.find(".", end)` gives the position of the next full stop after the mention, or −1 when there is none; the list comprehension keeps the positions that lie after the mention, and `min` takes the nearest. `low.rfind(". ", 0, start)` looks backwards for the start of the sentence. `any(w in after for w in NEGATION)` asks whether any trigger occurs in the scope. In the second summary, the window of 40 characters after "ht" reaches the (-) of the next diagnosis.

In [ ]:
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}


def lab_value(text, name):
    """The first number after a laboratory name, for example egfr 54 or hba1c %7,8; None when it is missing."""
    m = re.search(name + r"\s*%?\s*(\d+(?:,\d+)?)", tr_lower(text))
    return float(m.group(1).replace(",", ".")) if m else None


def rule_labels(text, window=None):
    """The mentions with their assertion, the drugs and two laboratory values."""
    labels = assertion_labels(text, window)
    low = tr_lower(text)
    for k, label in enumerate(LABELS):
        if any(re.search(r"(?<!\w)" + re.escape(w), low) for w in EVIDENCE.get(label, [])):
            labels[k] = 1
    egfr, hba1c = lab_value(text, "egfr"), lab_value(text, "hba1c")
    if egfr is not None and egfr < 60:
        labels[LABELS.index("CKD")] = 1
    if hba1c is not None and hba1c >= 6.5:
        labels[LABELS.index("DM")] = 1
    return labels

for name, mask in (("test, hospital A", te), ("external, hospital B", ex)):
    print(name)
    print(label_table(y[mask], np.array([rule_labels(t) for t in texts[mask]])))
print("Case VC-2444:", dict(zip(LABELS, rule_labels(texts[case]))))

**Python note.** `EVIDENCE` lists the drugs that point to a diagnosis. The pattern `(?<!\w)` + the name finds a drug as a whole word or as the start of a word, so that "insülin" also matches "insülin glarjin". `lab_value` reads the first number after a name, with an optional percent sign and the Turkish decimal comma, and returns `None` when the name is missing; `if egfr is not None and egfr < 60:` checks both. `rule_labels` starts from `assertion_labels` and sets the evidence on top of it.

**Observe and reflect.** The rule misses hypertension in about one patient of seven in hospital A. Print some of these summaries. Which words or drugs would you add, and which of your additions would also raise false alarms?

**Exercise C.** How many mentions of diagnoses in the 600 test summaries of hospital A does the rule of the course call negated?

In [ ]:
data = exercise_table("C")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = 0
for t in texts[te]:
    ms = find_terms(t)
    answer += sum(assertion(t, s, e, next_other(ms, i, t)) == "negated" for i, (s, e, l) in enumerate(ms))
teach.check("Exercise C", answer)

#### TF-IDF and logistic regression

*Lecture page: [Bag of words and TF-IDF: A summary as a vector](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#bag-of-words-and-tf-idf)*

A model learns the diagnoses from the training summaries. Every summary becomes a vector of TF-IDF weights of its words and pairs of words; a logistic regression for each diagnosis learns a weight for every term, and its threshold is chosen on the validation summaries.

**From the formulas to the code.** $\mathrm{idf}_t = \ln((1 + N)/(1 + \mathrm{df}_t)) + 1$: `vectorizer.idf_`. The frequency of a term enters as $1 + \ln \mathrm{tf}$ with `sublinear_tf=True`, and every vector is scaled to length 1. Single words and pairs of words: `ngram_range=(1, 2)`. Terms of at least two training summaries: `min_df=2`. The vocabulary and the weights are learned with `fit_transform` on the training summaries only and applied with `transform` to the others.

**Start cell.** For the section «TF-IDF and logistic regression», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «TF-IDF and logistic regression»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
    return text.replace("I", "ı").replace("İ", "i").lower()
def words(text):
    """The words of a text in lower case: runs of letters and digits."""
    return re.findall(r"\w+", tr_lower(text))
def se_ppv_f1(y_true, y_pred):
    """Sensitivity, positive predictive value and F1 of a yes-or-no decision."""
    y_true, y_pred = np.asarray(y_true, dtype=bool), np.asarray(y_pred, dtype=bool)
    tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
    se = tp / max(1, tp + fn)
    ppv = tp / max(1, tp + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    return se, ppv, f1
def label_table(y_true, y_pred):
    """One row per diagnosis: prevalence, sensitivity, PPV and F1; the last row is the mean F1 (macro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["diagnosis", "with the diagnosis", "sensitivity", "PPV", "F1"])
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table

**On the lecture page:** [Worked example: TF-IDF of three short summaries](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-tf-idf-of-three-short-summaries). The cell below computes the weights of the same three summaries.

In [ ]:
# The worked example of the lecture: three short summaries
mini = ["Atriyal fibrilasyon izlenmedi.", "Atriyal fibrilasyon. Apiksaban başlandı.", "Hipertansiyon. Amlodipin başlandı."]
mini_vectorizer = TfidfVectorizer(preprocessor=tr_lower, token_pattern=r"(?u)\b\w+\b")
X_mini = mini_vectorizer.fit_transform(mini)
print(pd.DataFrame(X_mini.toarray().round(3), columns=mini_vectorizer.get_feature_names_out()))
print(dict(zip(mini_vectorizer.get_feature_names_out(), mini_vectorizer.idf_.round(3))))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c154)*

**Python note.** `TfidfVectorizer` does the whole computation of the lecture: It finds the terms with the pattern `token_pattern`, here every run of letters and digits after `tr_lower`, counts them, multiplies by the idf and scales every row to length 1. `fit_transform` returns a *sparse* matrix that stores only the values that are not 0; `.toarray()` turns it into an ordinary array to print. The first row gives 0.518, 0.518 and 0.681, as in the worked example.

In [ ]:
vectorizer = TfidfVectorizer(preprocessor=tr_lower, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vectorizer.fit_transform(texts[tr])
X_val, X_test, X_ext = vectorizer.transform(texts[va]), vectorizer.transform(texts[te]), vectorizer.transform(texts[ex])
models, thresholds = {}, {}
GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
for k, label in enumerate(LABELS):
    models[label] = LogisticRegression(C=10, max_iter=2000).fit(X_train, y[tr, k])
    p_val = models[label].predict_proba(X_val)[:, 1]
    f1_val = [se_ppv_f1(y[va, k], p_val >= t)[2] for t in GRID]
    thresholds[label] = float(GRID[int(np.argmax(f1_val))])


def tfidf_labels(X):
    """Yes or no for every diagnosis: the probability of its model against its threshold from validation."""
    return np.column_stack([(models[label].predict_proba(X)[:, 1] >= thresholds[label]).astype(int) for label in LABELS])

print(X_train.shape, "terms;", "thresholds:", thresholds)
for name, X, mask in (("test, hospital A", X_test, te), ("external, hospital B", X_ext, ex)):
    print(name)
    print(label_table(y[mask], tfidf_labels(X)))

**Python note.** The loop trains one `LogisticRegression` for each diagnosis on `X_train` and the column of that diagnosis, `y[tr, k]`. `C=10` is the inverse of the strength of the regularisation of Week 2: a larger C lets the weights grow. `predict_proba(X_val)[:, 1]` gives the probability of the diagnosis for every validation summary, and the threshold with the highest F1 on the grid from 0.05 to 0.95 is kept in `thresholds`. `tfidf_labels` applies the six models and thresholds to any matrix.

In [ ]:
names = vectorizer.get_feature_names_out()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, label in zip(axes, ["AF", "HT", "CKD"]):
    w = models[label].coef_[0]
    top = np.argsort(w)[-8:]
    ax.barh(names[top], w[top], color="#1F5F8B")
    ax.set_title(label)
plt.tight_layout()
plt.show()

**Observe and reflect.** Look at the heaviest terms of atrial fibrillation, hypertension and chronic kidney disease. Which of them would a cardiologist accept as evidence, and which only describe how hospital A writes? Why does "egfr 57" carry weight, and what happens to a patient with an eGFR of 56?

**Exercise D.** Repeat the model with single words only, `ngram_range=(1, 1)`, and everything else unchanged. What is its macro F1 on the summaries of hospital B? Keep it with three decimals.

In [ ]:
data = exercise_table("D")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
v1 = TfidfVectorizer(preprocessor=tr_lower, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 1), min_df=2, sublinear_tf=True)
X1_train, X1_val, X1_ext = v1.fit_transform(texts[tr]), v1.transform(texts[va]), v1.transform(texts[ex])
f1s = []
for k, label in enumerate(LABELS):
    m = LogisticRegression(C=10, max_iter=2000).fit(X1_train, y[tr, k])
    p_val = m.predict_proba(X1_val)[:, 1]
    t = GRID[int(np.argmax([se_ppv_f1(y[va, k], p_val >= g)[2] for g in GRID]))]
    f1s.append(se_ppv_f1(y[ex, k], m.predict_proba(X1_ext)[:, 1] >= t)[2])
answer = round(float(np.mean(f1s)), 3)
teach.check("Exercise D", answer)

**Lemmatisation: one word, many forms.** *Lecture page: [Tokenization, lemmatization](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#tokens-cutting-text-into-pieces)*

Turkish adds suffixes to a root, so "yetmezlik", "yetmezliği" and "yetmezliğinin" are three columns of the bag of words. A *lemmatiser* brings every form back to its dictionary form, its *lemma*. The cells below use a small lemmatiser of the course for the clinical words of the archive. It looks for the longest stem of its dictionary at the start of a word and accepts the rest only when it is a chain of Turkish suffixes. It also knows that a final k becomes ğ before a vowel and that "ritim" and "kayıt" lose a vowel ("ritmi", "kaydı"). The TF-IDF model of this section is then trained again on the lemmas.

**From the formulas to the code.** The stems: `STEMS`, built from every lemma, its softened form and its form without the vowel. The rest of the word: `SUFFIX.fullmatch(...)`. The summaries as lemmas: `lemma_text`.

**On the lecture page:** [Worked example: The lemma of "yetmezliğinin"](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-the-lemma-of-yetmezli-inin). The cell below finds the lemma of the same word and of the other words of the archive.

In [ ]:
LEMMAS = ["tedavi", "öykü", "poliklinik", "kontrol", "ritim", "hastalık", "fraksiyon", "fonksiyon", "ağrı", "darlık", "yetersizlik",
          "yetmezlik", "enfarktüs", "servis", "anjiyografi", "ödem", "yakınma", "ölçüm", "kardeş", "anne", "baba", "derivasyon",
          "yükselme", "kayıt", "cerrahi", "bacak", "yıl", "tanı", "şüphe", "segment", "proksimal", "distal", "stent", "girişim", "kalp", "damar",
          "diyabet", "hipertansiyon", "fibrilasyon", "arter", "muayene", "sendrom", "ventrikül", "aritmi", "böbrek", "ekokardiyografi"]
SOFT = {"k": "ğ", "p": "b", "ç": "c", "t": "d"}          # yetmezlik → yetmezliği
DROPPED = {"ritim": "ritm", "kayıt": "kayd"}             # ritim → ritmi, kayıt → kaydı
SUFFIX = re.compile(r"(?:l[ae]r)?(?:s?[ıiuü]|l[ae]r[ıi])?(?:n?d[ae]n?|n?t[ae]n?|y?[ae]|n[ae]|y?l[ae]|n?[ıiuü]n|y?[ıiuü]|n[ıiuü]|[ıiuü]?yl[ae]|[ıiuü]?nd[ae]|ki)?")
STEMS = {}
for lem in LEMMAS:
    STEMS[lem] = lem
    STEMS[DROPPED.get(lem, lem)] = lem
    if lem[-1] in SOFT:
        STEMS[lem[:-1] + SOFT[lem[-1]]] = lem
STEM_ORDER = sorted(STEMS, key=len, reverse=True)


def lemma(word):
    """The lemma of a word in lower case: the longest stem of the dictionary that the word starts with, when the rest is a chain of Turkish suffixes."""
    for stem in STEM_ORDER:
        if word.startswith(stem) and SUFFIX.fullmatch(word[len(stem):]):
            return STEMS[stem]
    return word


def lemma_text(text):
    """A text with every word replaced by its lemma."""
    return " ".join(lemma(w) for w in words(text))

for w in ["yetmezliğinin", "ritmi", "kaydında", "anjiyografide", "darlığa", "segmentinde", "hastane"]:
    print(f"{w:16} → {lemma(w)}")
word_forms = Counter(w for t in texts[tr] for w in words(t))
print("Word forms of the training summaries:", len(word_forms),
      "→ lemmas:", len({lemma(w) for w in word_forms}))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c164)*

**Python note.** `sorted(STEMS, key=len, reverse=True)` puts the longest stems first, so that "hastalığ" is tried before "hasta" and "hastalığı" becomes "hastalık". `SUFFIX.fullmatch(rest)` is true only when the whole rest of the word matches the pattern of suffixes: an optional plural (`l[ae]r`), an optional possessive and an optional case ending. A word that matches no stem stays as it is, so "hastane" is not cut to "hasta".

In [ ]:
lemma_texts = np.array([lemma_text(t) for t in texts])
lemma_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
L_train = lemma_vectorizer.fit_transform(lemma_texts[tr])
L_val, L_test, L_ext = (lemma_vectorizer.transform(lemma_texts[m]) for m in (va, te, ex))
lemma_f1 = {"test": [], "external": []}
for k, label in enumerate(LABELS):
    lemma_model = LogisticRegression(C=10, max_iter=2000).fit(L_train, y[tr, k])
    p_val = lemma_model.predict_proba(L_val)[:, 1]
    t_best = GRID[int(np.argmax([se_ppv_f1(y[va, k], p_val >= g)[2] for g in GRID]))]
    for part, L_part, mask in (("test", L_test, te), ("external", L_ext, ex)):
        lemma_f1[part].append(se_ppv_f1(y[mask, k], lemma_model.predict_proba(L_part)[:, 1] >= t_best)[2])

macro_words = [round(float(np.mean([se_ppv_f1(y[m, k], tfidf_labels(X)[:, k])[2] for k in range(len(LABELS))])), 3) for X, m in ((X_test, te), (X_ext, ex))]
print("Terms:", X_train.shape[1], "words →", L_train.shape[1], "lemmas")
print("Macro F1, words:", dict(zip(["test", "external"], macro_words)))
print("Macro F1, lemmas:", {"test": round(float(np.mean(lemma_f1["test"])), 3), "external": round(float(np.mean(lemma_f1["external"])), 3)})

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c166)*

**Observe and reflect.** The archive is synthetic: Its generator writes most words in only a few forms, so lemmatisation changes the vocabulary little. Real reports use many more forms of the same word. Which words of a real discharge summary of your clinic would you add to the dictionary of the lemmatiser first?

#### Word vectors from the archive

*Lecture page: [Word vectors: Meaning from the company of words](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#word-vectors-meaning-from-the-company-of-words)*

Words get vectors from the words around them. The code counts how often two words occur together in the training summaries, compares the counts with chance by the PMI, keeps the positive values and shortens every row to 50 numbers. Two contexts are compared: a window of four words and the whole summary.

**From the formulas to the code.** $\mathrm{PMI}(a, b) = \ln(P(a, b)/(P(a)P(b)))$: `np.log(co / total) - np.log(np.outer(p_word, p_word))`. PPMI: `np.maximum(pmi, 0)`. The 50 main directions: `np.linalg.svd(ppmi)` and `U[:, :50] * S[:50]`. $\cos(\mathbf{u}, \mathbf{v})$: `vectors @ vectors[i]`, since every vector has length 1.

**Start cell.** For the section «Word vectors from the archive», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Word vectors from the archive»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
from collections import Counter
archive = reports.archive()
texts = archive["text_tr"].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
    return text.replace("I", "ı").replace("İ", "i").lower()
def words(text):
    """The words of a text in lower case: runs of letters and digits."""
    return re.findall(r"\w+", tr_lower(text))

**On the lecture page:** [Worked example: PMI and cosine with small numbers](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-pmi-and-cosine-with-small-numbers). The cell below computes the PPMI values of the words of the archive, shortens their rows by the singular value decomposition and compares the vectors by their cosine.

In [ ]:
def word_vectors(token_lists, window=None, min_count=5, dimensions=50):
    """Word vectors from co-occurrence: PPMI and SVD. With window=4 two words co-occur when they stand at most four words apart; with window=None, when they appear in the same report."""
    counts = Counter(w for ws in token_lists for w in ws)
    vocab = sorted(w for w, c in counts.items() if c >= min_count and not w.isdigit())
    index = {w: i for i, w in enumerate(vocab)}
    co = np.zeros((len(vocab), len(vocab)))
    for ws in token_lists:
        ids = [index.get(w) for w in ws]
        if window is None:
            present = sorted({i for i in ids if i is not None})
            co[np.ix_(present, present)] += 1
            continue
        for i, a in enumerate(ids):
            if a is None:
                continue
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i and ids[j] is not None:
                    co[a, ids[j]] += 1
    np.fill_diagonal(co, 0)
    total = co.sum()
    p_word = co.sum(axis=1) / total
    with np.errstate(divide="ignore"):
        pmi = np.log(co / total) - np.log(np.outer(p_word, p_word))
    ppmi = np.where(co > 0, np.maximum(pmi, 0), 0.0)
    U, S, _ = np.linalg.svd(ppmi)
    vectors = U[:, :dimensions] * S[:dimensions]
    return vocab, vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


def nearest(word, vocab, vectors, k=5):
    """The k words whose vectors have the largest cosine with the vector of the word."""
    cos = vectors @ vectors[vocab.index(word)]
    order = [i for i in np.argsort(-cos) if vocab[i] != word][:k]
    return [(vocab[i], round(float(cos[i]), 3)) for i in order]

train_tokens = [words(t) for t in texts[tr]]
vocab_w, vectors_w = word_vectors(train_tokens, window=4)
vocab_r, vectors_r = word_vectors(train_tokens, window=None)
print(len(vocab_w), "words with a vector")
for word in ["apiksaban", "hipertansiyon", "furosemid"]:
    print(f"{word:14} window:", nearest(word, vocab_w, vectors_w, 4))
    print(f"{'':14} summary:", nearest(word, vocab_r, vectors_r, 4))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c171)*

**Python note.** `Counter` counts the words of all training summaries, and the vocabulary keeps those that occur at least five times and are not numbers. `co` is a square table with one row and one column per word; the two nested loops add 1 for every pair of words that stand at most `window` words apart. With `window=None`, `np.ix_(present, present)` adds 1 to every pair of words of the same summary at once. `np.errstate(divide="ignore")` silences the warning for the logarithm of 0, whose results are then set to 0.

In [ ]:
show = ["apiksaban", "rivaroksaban", "dabigatran", "düzensiz", "metformin", "gliklazid", "furosemid", "spironolakton", "dekompanse",
        "amlodipin", "ramipril", "hipertansiyon", "hipertansyon", "ht", "egfr", "kreatinin"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (vocab_x, vectors_x, title) in zip(axes, [(vocab_w, vectors_w, "window of four words"),
                                                  (vocab_r, vectors_r, "whole summary")]):
    M = vectors_x[[vocab_x.index(w) for w in show]]
    M = M - M.mean(axis=0)
    xy = M @ np.linalg.svd(M, full_matrices=False)[2][:2].T
    ax.scatter(xy[:, 0], xy[:, 1], s=12, color="#9B1B1B")
    for (a, b), w in zip(xy, show):
        ax.annotate(w, (a, b), fontsize=8)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**Observe and reflect.** With the window, the anticoagulants lie together and apart from "düzensiz"; with the whole summary, "düzensiz" joins them. Which of the two spaces would help a model that must recognise a misspelled drug, and which a model that must guess a diagnosis from the drugs?

**Exercise E.** What is the cosine between the vectors of "apiksaban" and "düzensiz" when the whole summary is the context? Keep it with three decimals.

In [ ]:
data = exercise_table("E")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(vectors_r[vocab_r.index("apiksaban")] @ vectors_r[vocab_r.index("düzensiz")]), 3)
teach.check("Exercise E", answer)

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#last-two-hours-advanced-clinical-nlp-applications) and continue reading at «Last two hours: Advanced clinical NLP applications». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: Mini project: STEMI, AF and heart failure with an NER pipeline

*Lecture page: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#stop-2)*

This stop is the mini project of the session. A pretrained Turkish language model from the Hugging Face Hub reads the summaries here. Its tokenizer shows first what the capital letters of hospital B do to a model. The model is then fine-tuned to find diagnoses, drugs and procedures, and its entities, with the assertion rule and the drugs of Stop 1, give the six diagnoses of the summaries of hospital B. The last step adds STEMI, read from the acute coronary syndrome and the ST elevation of the summaries.

**At the end of this stop:** The share of unknown tokens in the summaries of hospital B written as they are (percent 19.5), the F1 of the entities of the fine-tuned model on the test set (about 0.981) the macro F1 of the six diagnoses of hospital B from its entities (about 0.754) and the F1 of STEMI on the test set of hospital A (1.0).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 2 · Python basics

*Lecture page: [Named Entity Recognition (NER): Drugs, diagnoses, procedures](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#named-entities-diagnoses-drugs-and-procedures)*

The ready application of this stop tries three lower cases on the tokens of the archive, gives every token of a summary the tag of its entity, judges the entities by exact match and links a misspelled mention to the most similar word of the dictionary. The examples below introduce these constructs on short texts: a dictionary of functions, the positions of the words of a text, sets of entities and the similarity of two strings.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 2 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower

# repeated from the earlier cells: the data and the code that the cells below use
import difflib
import re

In [ ]:
# Python practice: a small example for learning the language
forms = {"as written": lambda t: t, "Python's lower()": str.lower, "Turkish lower case": tr_lower}
word = "KALP YETMEZLİĞİ"
for name, f in forms.items():
    print(f"{name:26} {f(word)}  ({len(f(word))} characters)")

**Python note.** A dictionary can hold functions as its values. `lambda t: t` writes a small function without a name that returns its input unchanged, and `str.lower` is Python's own method, used here as a function. The loop calls each function on the same word: Python's `lower()` turns İ into two characters, so the word grows by one character, while the Turkish lower case of the course keeps its length. The ready application compares the three forms in the same way on the tokens of the whole archive.

In [ ]:
# Python practice: a small example for learning the language
text = "Kalp yetmezliği için furosemid başlandı."
entities = [{"start": 0, "end": 15, "label": "DIAG"}, {"start": 21, "end": 30, "label": "DRUG"}]
offsets = [m.span() for m in re.finditer(r"\w+", text)]   # (start, end) of every word
tags = []
for start, end in offsets:
    tag = "O"
    for e in entities:
        if e["start"] <= start < e["end"]:
            tag = ("B-" if start == e["start"] else "I-") + e["label"]
    tags.append(tag)
for (start, end), tag in zip(offsets, tags):
    print(f"{text[start:end]:12} {start:2}-{end:2}  {tag}")

**Python note.** `m.span()` gives the position of the first character of a match and that of the character after its last, so `text[start:end]` cuts the word back out of the text. An entity is stored in the same way, with its start, its end and its type. The loop gives every word the tag of the entity in which its first character lies: B- for the first word of the entity, I- for the words that follow it and O outside every entity. These are the tags of the BIO scheme, and the function `encode` of the ready application tags the tokens of the model in the same way.

In [ ]:
# Python practice: a small example for learning the language
gold = {(0, 15, "DIAG"), (21, 30, "DRUG"), (45, 52, "PROC")}                    # the annotated entities
found = {(0, 15, "DIAG"), (21, 30, "DRUG"), (45, 49, "PROC"), (60, 64, "DIAG")}   # the entities that a model found
tp, fp, fn = len(gold & found), len(found - gold), len(gold - found)
print("TP", tp, "FP", fp, "FN", fn)
print("precision", round(tp / (tp + fp), 3), "recall", round(tp / (tp + fn), 3), "F1", round(2 * tp / (2 * tp + fp + fn), 3))

**Python note.** A *set* holds every value once, between curly brackets; here its values are the entities, each a tuple of start, end and type. `gold & found` keeps the entities that are in both sets, the true positives, `found - gold` those that only the model found and `gold - found` those that it missed. An entity counts only when its start, its end and its type all agree: The procedure that the model ends at character 49 instead of 52 is a false positive and a false negative at once. The function `span_scores` of the ready application counts the entities in the same way.

In [ ]:
# Python practice: a small example for learning the language
dictionary = ["hipertansiyon", "diyabet", "kalp yetmezliği"]
for mention in ["hipertansyon", "diyebet", "nefes darlığı"]:
    score, best = max((difflib.SequenceMatcher(None, mention, w).ratio(), w) for w in dictionary)
    print(f"{mention:14} → {best:16} {score:.2f}", "linked" if score >= 0.8 else "not linked")

**Python note.** `difflib.SequenceMatcher(None, a, b).ratio()` compares two strings letter by letter: It gives 1 for the same text and less the more letters differ. The generator inside `max(...)` builds a pair (score, word) for every word of the dictionary, and `max` compares the pairs by their first value, so it returns the most similar word together with its score. A misspelled mention is linked to a diagnosis only when the score reaches 0.8, so "nefes darlığı" stays without one. The function `normalise` of the ready application links the entities of the model to the six diagnoses in the same way.

**Exercise 2.1.** In the sentence "Akut koroner sendrom nedeniyle koroner anjiyografi yapıldı.", the diagnosis "Akut koroner sendrom" takes the characters 0 to 20 and the procedure "koroner anjiyografi" the characters 31 to 50. How many words of the sentence get an I- tag?

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
text = "Akut koroner sendrom nedeniyle koroner anjiyografi yapıldı."
entities = [{"start": 0, "end": 20, "label": "DIAG"}, {"start": 31, "end": 50, "label": "PROC"}]
answer = 0
for start, end in [m.span() for m in re.finditer(r"\w+", text)]:
    for e in entities:
        if e["start"] < start < e["end"]:
            answer += 1
teach.check("Exercise 2.1", answer)

**Exercise 2.2.** The annotated entities of a summary are (0, 15, "DIAG"), (21, 30, "DRUG"), (40, 52, "DIAG") and (60, 70, "PROC"). A model finds (0, 15, "DIAG"), (21, 30, "DRUG"), (40, 48, "DIAG") and (75, 84, "DRUG"). What is the F1 of the entities by exact match? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gold = {(0, 15, "DIAG"), (21, 30, "DRUG"), (40, 52, "DIAG"), (60, 70, "PROC")}
found = {(0, 15, "DIAG"), (21, 30, "DRUG"), (40, 48, "DIAG"), (75, 84, "DRUG")}
tp, fp, fn = len(gold & found), len(found - gold), len(gold - found)
answer = round(2 * tp / (2 * tp + fp + fn), 3)
teach.check("Exercise 2.2", answer)

**«Stop 2 · Python basics» is complete.** Next comes «Stop 2 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#summaries-and-large-language-models-as-clinical-assistants) and continue reading at «Automatic summarisation of radiology reports».

### Stop 2 · Prompt and application

This part has four prompts: «The tokens of a pretrained Turkish model», «Fine-tuning the model for entities», «From entities to the six diagnoses of hospital B» and «STEMI from the course and the ECG of a summary». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-06/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 2 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

import torch
from vclinic import reports
import re

if "archive" not in globals():
    archive = reports.archive()
    LABELS = list(reports.LABELS)
    texts = archive["text_tr"].to_numpy()
    y = archive[LABELS].to_numpy()
    split = archive["split"].to_numpy()
    tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


    def tr_lower(text):
        """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """The words of a text in lower case: runs of letters and digits."""
        return re.findall(r"\w+", tr_lower(text))
_COLS = ["HF", "AF", "CAD", "HT", "DM", "CKD"]
_SETS = {"training": "training", "validation": "validation", "test": "test", "external": "external"}


def _summaries(mask, with_set=False, with_entities=False):
    """The summaries of a set with the six diagnoses of the registry, as the prompts describe them."""
    t = pd.DataFrame({"summary": texts[mask]})
    if with_set:
        t.insert(0, "set", [_SETS[s] for s in split[mask]])
    if with_entities:
        t["entities"] = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    for k, c in enumerate(_COLS):
        t[c] = y[mask, k]
    return t.reset_index(drop=True)


def _course_ner():
    """The entities that the model of the course finds in the external summaries (trained once, a few minutes)."""
    if "_ner_cache" in globals():
        return _ner_cache
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    torch.manual_seed(2026)
    np.random.seed(2026)
    MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
    tokenizer = AutoTokenizer.from_pretrained(MODEL)

    TAGS = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
    MAX_LENGTH = 384


    def encode(text, entities):
        """Tokens of a report with one tag each: B- for the first token of an entity, I- for the others, O outside; -100 for the special tokens."""
        enc = tokenizer(tr_lower(text), return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
        tags = []
        for start, end in enc["offset_mapping"]:
            if start == end:
                tags.append(-100)
                continue
            tag = "O"
            for e in entities:
                if e["start"] <= start < e["end"]:
                    tag = ("B-" if start == e["start"] else "I-") + e["label"]
                    break
            tags.append(TAGS.index(tag))
        return enc["input_ids"], tags, enc["offset_mapping"]


    def spans(tag_ids, offsets):
        """Entities from the tags of the tokens: (start, end, type) in characters of the report."""
        found, current = [], None
        for t, (start, end) in zip(tag_ids, offsets):
            if start == end:
                continue
            tag = TAGS[t]
            if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
                if current:
                    found.append(tuple(current))
                current = [start, end, tag[2:]]
            elif tag.startswith("I-"):
                current[1] = end
            else:
                if current:
                    found.append(tuple(current))
                current = None
        if current:
            found.append(tuple(current))
        return found


    def batches(items, size, shuffle=False):
        """Groups of reports padded to the same length, for training and prediction."""
        order = np.random.permutation(len(items)) if shuffle else np.arange(len(items))
        for s in range(0, len(items), size):
            group = [items[i] for i in order[s:s + size]]
            width = max(len(g[0]) for g in group)
            ids = torch.tensor([g[0] + [0] * (width - len(g[0])) for g in group])
            tags = torch.tensor([g[1] + [-100] * (width - len(g[1])) for g in group])
            mask = torch.tensor([[1] * len(g[0]) + [0] * (width - len(g[0])) for g in group])
            yield ids, tags, mask

    EPOCHS, BATCH, LEARNING_RATE, SEED = 4, 16, 3e-4, 2026
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    train_items = [encode(t, e)[:2] for t, e in zip(texts[tr], archive["entities_tr"][tr])]
    ner_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS))
    optimizer = torch.optim.AdamW(ner_model.parameters(), lr=LEARNING_RATE)
    for epoch in range(EPOCHS):
        ner_model.train()
        total = 0.0
        for ids, tags, mask in batches(train_items, BATCH, shuffle=True):
            loss = ner_model(input_ids=ids, attention_mask=mask, labels=tags).loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(ids)
        print(f"epoch {epoch + 1}: mean loss {total / len(train_items):.4f}")


    def predict_spans(text_list):
        """Entities found by the model in every report of a list."""
        ner_model.eval()
        out = []
        with torch.no_grad():
            for text in text_list:
                ids, _, offsets = encode(text, [])
                logits = ner_model(input_ids=torch.tensor([ids])).logits[0]
                out.append(spans(logits.argmax(-1).tolist(), offsets))
        return out
    globals()["_ner_cache"] = predict_spans(list(texts[ex]))
    return _ner_cache

print(len(archive), "summaries; hospital B:", int(ex.sum()))

#### Prompt 2.1: The tokens of a pretrained Turkish model

**Goal.** See how a pretrained Turkish model cuts the summaries of hospital B into tokens, and what the case of the letters changes.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to use a pretrained Turkish language model on discharge summaries written partly in capitals, and to know whether the model knows their words.  
> Data: The notebook holds a table named "data" with the 600 discharge summaries of hospital B of the virtual clinic, a synthetic registry, written in Turkish in a short style with abbreviations in capitals. Each row is a patient, and "summary" holds the discharge summary.  
> Task: Load the tokenizer of the model "ytu-ce-cosmos/turkish-tiny-bert-uncased" from the Hugging Face Hub. Cut every summary into tokens three times: as it is written, after the usual lower case of the programming language, and after the Turkish lower case (first I becomes ı and İ becomes i, then the other letters become small). For each of the three, count the tokens without the special tokens at the start and the end, and compute the percentage of them that are the unknown token. Also show the tokens of the first summary in all three forms.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of tokens and the percentage of unknown tokens for the three forms, and keep the percentage for the text as written, with one decimal, under the name "answer".  
> Test: The Turkish lower case must leave no unknown token, and the first summary must give more tokens after the usual lower case than after the Turkish one or the same number.  

In [ ]:
# Data of Prompt 2.1: the table "data" that the prompt describes
data = pd.DataFrame({"summary": texts[ex]})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.1
teach.check_step("Prompt 2.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"summary": texts[ex]})
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ytu-ce-cosmos/turkish-tiny-bert-uncased")
forms = {"as written": lambda t: t, "usual lower case": lambda t: t.lower(),
          "Turkish lower case": lambda t: t.replace("I", "ı").replace("İ", "i").lower()}
shares = {}
for name, f in forms.items():
    ids = [tokenizer(f(t))["input_ids"][1:-1] for t in data["summary"]]
    n = sum(len(i) for i in ids)
    unknown = sum(i.count(tokenizer.unk_token_id) for i in ids)
    shares[name] = 100 * unknown / n
    print(f"{name}: {n} tokens, {unknown} [UNK] ({shares[name]:.1f} %)")
    print("   ", tokenizer.tokenize(f(data["summary"].iloc[0]))[:25])
answer = round(shares["as written"], 1)
teach.check_step("Prompt 2.1", "answer", globals())

**What the result shows.** Written as it is, 19.5 percent of the tokens of hospital B are unknown: The vocabulary of the uncased model has no capital letters. The usual lower case removes the unknown tokens but cuts every word with İ wrongly, because it adds a dot after the i; only the Turkish lower case gives the words of the vocabulary.

#### Prompt 2.2: Fine-tuning the model for entities

**Goal.** Teach the pretrained model to tag diagnoses, drugs and procedures. The training takes a few minutes.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to adapt a small pretrained Turkish language model so that it finds the diagnoses, drugs and procedures of discharge summaries, and to know how exactly it finds them in summaries it has not seen.  
> Data: The notebook holds a table named "data" with 3,000 discharge summaries of the virtual clinic, a synthetic registry. Each row is a patient: "set" gives the set ("training", 1,800 summaries of hospital A; "test", 600 of hospital A; "external", 600 of hospital B), "summary" holds the discharge summary in Turkish, and "entities" lists its annotated entities, each as (first character, last character plus one, type), counted from 0, with the types "DIAG" for a diagnosis, "DRUG" for a drug and "PROC" for a procedure; the six columns "HF", "AF", "CAD", "HT", "DM" and "CKD" hold 0 or 1, with 1 when the registry records heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease, and 0 otherwise.  
> Task: Use the model "ytu-ce-cosmos/turkish-tiny-bert-uncased" from the Hugging Face Hub with a layer that gives every token one of seven tags: outside an entity, or the beginning or the inside of a diagnosis, a drug or a procedure. Bring every summary to the Turkish lower case (first I becomes ı and İ becomes i), which keeps the positions of the characters, and cut it into at most 384 tokens. A token gets the tag of the entity in which its first character lies: the beginning tag when it starts the entity, the inside tag otherwise. Fine-tune the model on the training summaries for 4 epochs, in shuffled groups of 16 summaries, with the optimiser AdamW and a learning rate of 0.0003. Then tag the test summaries and the external summaries, join the tokens of every entity back into a stretch of characters, and compute the precision, the recall and the F1 score of the entities, where a predicted entity counts only when its type, its first character and its last character match an annotated one. Fix the random seed to 2026 so that the result can be repeated.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the loss of every epoch and a table with the precision, the recall and the F1 score for all entities and for each type, on the test and on the external summaries, and keep the F1 score of all entities on the test summaries, with three decimals, under the name "answer".  
> Test: The loss must fall from the first to the last epoch, and the F1 score must lie between 0 and 1.  

In [ ]:
# Data of Prompt 2.2: the table "data" that the prompt describes
data = _summaries(tr | te | ex, with_set=True, with_entities=True)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.2
teach.check_step("Prompt 2.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.2 (run it only if the code of the assistant does not pass the check)
data = _summaries(tr | te | ex, with_set=True, with_entities=True)
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

torch.manual_seed(2026)
np.random.seed(2026)
tags = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
tokenizer = AutoTokenizer.from_pretrained("ytu-ce-cosmos/turkish-tiny-bert-uncased")


def encode(text, entities):
    enc = tokenizer(text.replace("I", "ı").replace("İ", "i").lower(), return_offsets_mapping=True, truncation=True, max_length=384)
    labels = []
    for a, b in enc["offset_mapping"]:
        if a == b:
            labels.append(-100)
            continue
        tag = next((("B-" if a == s else "I-") + t for s, e, t in entities if s <= a < e), "O")
        labels.append(tags.index(tag))
    return enc["input_ids"], labels, enc["offset_mapping"]


def to_spans(tag_ids, offsets):
    found, current = [], None
    for t, (a, b) in zip(tag_ids, offsets):
        if a == b:
            continue
        tag = tags[t]
        if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
            if current:
                found.append(tuple(current))
            current = [a, b, tag[2:]]
        elif tag.startswith("I-"):
            current[1] = b
        else:
            if current:
                found.append(tuple(current))
            current = None
    if current:
        found.append(tuple(current))
    return found


train = data[data["set"] == "training"]
items = [encode(t, e)[:2] for t, e in zip(train["summary"], train["entities"])]
model = AutoModelForTokenClassification.from_pretrained("ytu-ce-cosmos/turkish-tiny-bert-uncased", num_labels=len(tags))
optimiser = torch.optim.AdamW(model.parameters(), lr=3e-4)
for epoch in range(4):
    model.train()
    order = np.random.permutation(len(items))
    total = 0.0
    for start in range(0, len(order), 16):
        group = [items[i] for i in order[start:start + 16]]
        w = max(len(g[0]) for g in group)
        ids = torch.tensor([g[0] + [0] * (w - len(g[0])) for g in group])
        lab = torch.tensor([g[1] + [-100] * (w - len(g[1])) for g in group])
        mask = torch.tensor([[1] * len(g[0]) + [0] * (w - len(g[0])) for g in group])
        loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        total += loss.item() * len(group)
    print("epoch", epoch + 1, "loss", round(total / len(items), 4))
model.eval()
rows = []
for s in ("test", "external"):
    part = data[data["set"] == s]
    predicted = []
    with torch.no_grad():
        for t in part["summary"]:
            ids, _, offsets = encode(t, [])
            predicted.append(to_spans(model(input_ids=torch.tensor([ids])).logits[0].argmax(-1).tolist(), offsets))
    for kind in (None, "DIAG", "DRUG", "PROC"):
        tp = fp = fn = 0
        for g, p in zip(part["entities"], predicted):
            g = {tuple(e) for e in g if kind in (None, e[2])}
            p = {e for e in p if kind in (None, e[2])}
            tp, fp, fn = tp + len(g & p), fp + len(p - g), fn + len(g - p)
        rows.append({"set": s, "type": kind or "all", "precision": round(tp / max(1, tp + fp), 3),
                         "recall": round(tp / max(1, tp + fn), 3), "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
table = pd.DataFrame(rows)
print(table)
answer = float(table.query("type == 'all'")["F1"].iloc[0])
teach.check_step("Prompt 2.2", "answer", globals())

**What the result shows.** The fine-tuned model finds the entities of the test set with an F1 of about 0.981. Its errors there lie at the edges of entities, where a Turkish suffix sticks to the name of a procedure, as in "koroner anjiyografida". On the summaries of hospital B, it finds most drugs but few diagnoses and no procedure: Hospital B writes them as abbreviations in its short style.

#### Prompt 2.3: From entities to the six diagnoses of hospital B

**Goal.** Turn the entities of the model into the six diagnoses of every summary of hospital B, with the assertion rule and the drugs of Stop 1.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to turn the diagnoses and drugs that a language model found in discharge summaries of another hospital into the six chronic diagnoses of every patient, and to know how well this works.  
> Data: The notebook holds a table named "data" with the 600 discharge summaries of hospital B of the virtual clinic, a synthetic registry, written in Turkish in a short style. Each row is a patient: "summary" holds the discharge summary; "entities" lists its annotated entities and "model entities" the entities that the tiny Turkish language model of the course, fine-tuned on the training summaries of hospital A, found in it, both as (first character, last character plus one, type), counted from 0, with the types "DIAG" for a diagnosis, "DRUG" for a drug and "PROC" for a procedure; the six columns "HF", "AF", "CAD", "HT", "DM" and "CKD" hold 0 or 1, with 1 when the registry records heart failure, atrial fibrillation, coronary artery disease, hypertension, diabetes mellitus and chronic kidney disease, and 0 otherwise.  
> Task: Use the column "model entities". Give every diagnosis entity to one of the six diagnoses by its words, as in Stop 1, and decide its assertion with the rule of Stop 1; a diagnosis is present when one of its entities is present. Add the diagnoses of the drug entities and of the values of eGFR and HbA1c, as in Stop 1. Compare the six decisions with the six columns of the registry and compute, for each diagnosis, the sensitivity, the positive predictive value and the F1 score (twice the true positives divided by twice the true positives plus the false positives plus the false negatives), and the mean of the six F1 scores, the macro F1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with one row for each diagnosis and the three values, and keep the macro F1, with three decimals, under the name "answer".  
> Test: Every value must lie between 0 and 1, and a summary without any entity must be negative for every diagnosis except chronic kidney disease and diabetes, which can come from the laboratory values.  

In [ ]:
# Data of Prompt 2.3: the table "data" that the prompt describes
data = _summaries(ex, with_entities=True)
data.insert(2, "model entities", _course_ner())   # a few minutes the first time: the model of the course is fine-tuned once
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.3
teach.check_step("Prompt 2.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.3 (run it only if the code of the assistant does not pass the check)
data = _summaries(ex, with_entities=True)
data.insert(2, "model entities", _course_ner())   # a few minutes the first time: the model of the course is fine-tuned once
terms = {"HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def score(decisions):
    rows = []
    for c in terms:
        y_true = data[c].to_numpy() == 1
        y_pred = np.array(decisions[c]) == 1
        tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
        rows.append({"diagnosis": c, "sensitivity": round(tp / max(1, tp + fn), 3),
                         "PPV": round(tp / max(1, tp + fp), 3), "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
    table = pd.DataFrame(rows)
    print(table)
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table

import difflib

ext = data
predicted = data["model entities"]
negation = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)", "şüphe", "olası", "ön tanı"]
relative = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
drugs = {"HF": ["furosemid", "sakubitril"], "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
           "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"], "HT": ["amlodipin", "hidroklorotiyazid"],
           "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"]}
all_words = [(w, c) for c, words in terms.items() for w in words]


def diagnosis_of(mention):
    for w, c in sorted(all_words, key=lambda x: -len(x[0])):
        if re.search(r"(?<!\w)" + re.escape(w) + r"(?!\w)", mention):
            return c
    score_best = max((difflib.SequenceMatcher(None, mention, w).ratio(), c) for w, c in all_words)
    return score_best[1] if score_best[0] >= 0.8 else None


decisions = {c: [] for c in terms}
for text, entities in zip(ext["summary"], predicted):
    low = turkish_lower(text)
    mentions = sorted((s0, e0, diagnosis_of(low[s0:e0])) for s0, e0, kind in entities if kind == "DIAG")
    mentions = [m for m in mentions if m[2]]
    found = set()
    for i, (s0, e0, c) in enumerate(mentions):
        next_other = next((m[0] for m in mentions[i + 1:] if m[2] != c), len(low))
        stop = min(x for x in (low.find(".", e0), low.find(";", e0), next_other, len(low)) if x >= e0)
        sentence_start = max(low.rfind(". ", 0, s0), low.rfind(": ", 0, s0)) + 1
        if any(w in low[sentence_start:s0] for w in relative) or any(w in low[e0:stop] for w in negation):
            continue
        found.add(c)
    for s0, e0, kind in entities:
        if kind in ("DRUG", "PROC"):
            found.update(c for c, words in drugs.items()
                           if any(re.search(r"(?<!\w)" + re.escape(w), low[s0:e0]) for w in words))
    egfr = re.search(r"egfr\s*%?\s*(\d+(?:,\d+)?)", low)
    if egfr and float(egfr.group(1).replace(",", ".")) < 60:
        found.add("CKD")
    hba1c = re.search(r"hba1c\s*%?\s*(\d+(?:,\d+)?)", low)
    if hba1c and float(hba1c.group(1).replace(",", ".")) >= 6.5:
        found.add("DM")
    for c in terms:
        decisions[c].append(int(c in found))
table = score(decisions)
answer = round(float(table["F1"].mean()), 3)
teach.check_step("Prompt 2.3", "answer", globals())

**What the result shows.** From the entities of the model, the six diagnoses of hospital B reach a macro F1 of about 0.754, against 0.708 for the TF-IDF model of Stop 1. Two kinds of error lower the result. The model misses most abbreviations of the diagnoses and every procedure. And when an entity takes in the sign, as in "KBY (-)", the assertion rule finds no "(-)" after it and counts the negated disease as present. The entities of the model, and the macro F1 with them, change from one training run to the next: Over six runs of the course with different seeds, the macro F1 lies between 0.613 and 0.792. The check therefore accepts a wide band around the result of the course.

#### Prompt 2.4: STEMI from the course and the ECG of a summary

**Goal.** Find the patients with a STEMI from what the summaries write about the acute coronary syndrome and the ST segment, and judge the result on both hospitals.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to find the patients with an ST-elevation myocardial infarction in the discharge summaries of two hospitals and to know how reliably a simple rule finds them.  
> Data: The notebook holds a table named "data" with the 1,200 discharge summaries of the test set of hospital A, 600 in full sentences, and of hospital B, 600 in a short style, written in Turkish. Each row is a patient: "set" gives the set ("test" or "external"), "summary" holds the discharge summary, and "STEMI" is 1 when the registry of the virtual clinic recorded an ST-elevation myocardial infarction and 0 otherwise.  
> Task: Bring every summary to small letters by the Turkish rules: First turn I into ı and İ into i, then make the other letters small. Call a summary STEMI when it mentions an acute coronary syndrome, "akut koroner sendrom" or the abbreviation "aks" as a whole word, without one of the words "dışlandı", "yok", "izlenmedi" or "saptanmadı" after it in the same sentence, and when it also contains "st yükselmesi". Compare with the column "STEMI" and compute, for each set, the number of patients with STEMI, the number found, the sensitivity, the positive predictive value and the F1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with one row for each set and these values, and keep the F1 of the set "test", with three decimals, under the name "answer".  
> Test: Every value must lie between 0 and 1, and the number of patients with STEMI of each set must equal the sum of the column "STEMI" in that set.  

In [ ]:
# Data of Prompt 2.4: the table "data" that the prompt describes
from vclinic import make_cohort
_reg = make_cohort(len(archive)).set_index("patient_id")
_m = te | ex
data = pd.DataFrame({"set": [_SETS[s] for s in split[_m]], "summary": texts[_m],
                     "STEMI": (_reg.loc[archive["patient_id"][_m], "stemi_territory"] != "none").astype(int).to_numpy()})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.4
teach.check_step("Prompt 2.4", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.4 (run it only if the code of the assistant does not pass the check)
from vclinic import make_cohort
_reg = make_cohort(len(archive)).set_index("patient_id")
_m = te | ex
data = pd.DataFrame({"set": [_SETS[s] for s in split[_m]], "summary": texts[_m],
                     "STEMI": (_reg.loc[archive["patient_id"][_m], "stemi_territory"] != "none").astype(int).to_numpy()})
rows = []
for set_name, part in data.groupby("set", sort=False):
    found = []
    for text in part["summary"]:
        low = text.replace("I", "ı").replace("İ", "i").lower()
        acs = False
        for m in re.finditer(r"akut koroner sendrom|(?<!\w)aks(?!\w)", low):
            stop = low.find(".", m.end())
            rest = low[m.end():stop if stop >= 0 else len(low)]
            if not any(w in rest for w in ["dışlandı", "yok", "izlenmedi", "saptanmadı"]):
                acs = True
        found.append(acs and "st yükselmesi" in low)
    true = part["STEMI"].to_numpy() == 1
    pred = np.array(found)
    tp, fp, fn = int((pred & true).sum()), int((pred & ~true).sum()), int((~pred & true).sum())
    rows.append({"set": set_name, "STEMI": int(true.sum()), "found": int(pred.sum()),
                     "sensitivity": round(tp / max(1, tp + fn), 3), "PPV": round(tp / max(1, tp + fp), 3),
                     "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
table = pd.DataFrame(rows)
print(table)
answer = float(table.loc[table["set"] == "test", "F1"].iloc[0])
teach.check_step("Prompt 2.4", "answer", globals())

**What the result shows.** The rule finds every STEMI of hospital A: Its summaries name the acute coronary syndrome in the course and the leads with ST elevation in the ECG. Hospital B writes only "AKS" in its short course and nothing about the ST segment, so no rule and no model can find a STEMI there. What a report does not contain cannot be extracted from it.

**«Stop 2 · Prompt and application» is complete.** Next comes «Stop 2 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#summaries-and-large-language-models-as-clinical-assistants) and continue reading at «Automatic summarisation of radiology reports».

### Stop 2 · Ready application

This part loads the tiny Turkish BERT of the course and the tokenizer of BioBERT from the Hugging Face Hub, fine-tunes the Turkish model for the entities of the summaries, turns its entities into the six diagnoses and runs the mini project, STEMI, AF and heart failure with an NER pipeline.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 2 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower, words, se_ppv_f1, next_other, lab_value

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import difflib
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
def label_table(y_true, y_pred):
    """One row per diagnosis: prevalence, sensitivity, PPV and F1; the last row is the mean F1 (macro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["diagnosis", "with the diagnosis", "sensitivity", "PPV", "F1"])
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Every mention of a diagnosis: (start, end, diagnosis), in the order of the text."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
def assertion(text, start, end, next_start, window=None):
    """The assertion of one mention: present, negated, possible or family.

    The scope after the mention ends at the end of the sentence or where the next mention of another diagnosis starts (next_start); with a window, it ends after that many characters instead."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}

#### A pretrained Turkish model

*Lecture page: [BERT / BioBERT: Loading from the Hugging Face Hub](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#pretrained-language-models-learning-a-language-before-a-task)*

The tiny Turkish BERT of the course has two layers, 128 numbers per token and a vocabulary of 32,000 pieces. This section counts the tokens of the archive with three kinds of lower case, looks at the size of the model and lets it fill gaps in clinical sentences.

**From the formulas to the code.** The share of unknown tokens is the count of `tokenizer.unk_token_id` divided by all tokens. The parameters of the model: `sum(p.numel() for p in model.parameters())`. The masked language model: `pipeline("fill-mask", model=MODEL)` with [MASK] in the sentence.

In [ ]:
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

forms = {"as written": lambda t: t, "Python's lower()": str.lower, "Turkish lower case": tr_lower}
for name, f in forms.items():
    for hospital, mask in (("A", tr), ("B", ex)):
        ids = [tokenizer(f(t))["input_ids"][1:-1] for t in texts[mask]]
        n = sum(len(i) for i in ids)
        unknown = sum(i.count(tokenizer.unk_token_id) for i in ids)
        print(f"{name:26} hospital {hospital}: {n / len(ids):6.1f} tokens per summary, [UNK] {100 * unknown / n:5.1f} %")
for word in ["KALP YETMEZLİĞİ", "PROKSİMAL", "hipertansyon"]:
    print(word, "→", tokenizer.tokenize(word), tokenizer.tokenize(word.lower()), tokenizer.tokenize(tr_lower(word)))

**Python note.** `tokenizer(text)["input_ids"][1:-1]` drops the first and the last token, [CLS] and [SEP]. `i.count(tokenizer.unk_token_id)` counts the unknown tokens of one summary. `tokenizer.tokenize(word)` gives the pieces as text instead of numbers. The dictionary `forms` holds three functions, so the same loop can try all three; `str.lower` is Python's own method, used here as a function.

In [ ]:
fill = pipeline("fill-mask", model=MODEL)
print("Parameters of the model:", sum(p.numel() for p in fill.model.parameters()))
for sentence in ["hasta [MASK] ağrısı yakınmasıyla başvurdu.", "ekg: sinüs [MASK].", "taburcu tedavisi: [MASK] 40 mg."]:
    print(sentence, [(g["token_str"], round(g["score"], 3)) for g in fill(sentence)])
for word in ["göğüs", "ritmi", "furosemid"]:
    print(word, "→", tokenizer.tokenize(word))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c184)*

**Observe and reflect.** The model fills the gaps with what it learned from general Turkish text. Where does "göğüs" stand among its guesses for the pain, and why might a model pretrained on cardiology reports put it first? A [MASK] stands for one token: Why can the model not propose "ritmi" for the ECG or "furosemid" for the drug?

**Exercise F.** How many tokens does a summary of hospital B have on average after the Turkish lower case, without the two special tokens? Keep the mean with one decimal.

In [ ]:
data = exercise_table("F")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(np.mean([len(tokenizer(tr_lower(t))["input_ids"]) - 2 for t in texts[ex]])), 1)
teach.check("Exercise F", answer)

**BioBERT from the Hugging Face Hub.** *Lecture page: [BERT / BioBERT: Loading from the Hugging Face Hub](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#pretrained-language-models-learning-a-language-before-a-task)*

BioBERT is the English BERT, trained further on biomedical articles. The cell below loads only its tokenizer from the Hugging Face Hub, a few hundred kilobytes, and compares how BioBERT and the Turkish model of the course cut an English and a Turkish clinical sentence and the summaries of the archive.

**From the formulas to the code.** Any model of the Hub loads with its name: `AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")`. BioBERT keeps capital letters, `do_lower_case=False`. The pieces per word: the number of tokens of a summary divided by the number of its words.

In [ ]:
bio_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2", do_lower_case=False)

print("Vocabulary: BioBERT", len(bio_tokenizer), "· Turkish model of the course", len(tokenizer))
for sentence in ["Immunoglobulin levels were normal.", "Kalp yetmezliğinin nedeni koroner arter hastalığıdır."]:
    print(sentence)
    print("   BioBERT:", bio_tokenizer.tokenize(sentence))
    print("   Turkish model:", tokenizer.tokenize(tr_lower(sentence)))
pieces = lambda tok, f: float(np.mean([len(tok.tokenize(f(t))) / len(words(t)) for t in texts[te][:200]]))
print("Pieces per word, 200 test summaries: BioBERT", round(pieces(bio_tokenizer, lambda t: t), 2),
      "· Turkish model", round(pieces(tokenizer, tr_lower), 2))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c190)*

**Python note.** `AutoTokenizer.from_pretrained(name)` downloads the files of the tokenizer once and keeps them in a folder of the computer; a second call reads them from there. `len(tokenizer)` is the size of the vocabulary. `lambda tok, f: ...` defines a small function without a name; here it takes a tokenizer and a function of lower case, so that the two models are compared with one line each.

**Observe and reflect.** BioBERT learned medicine from English articles, but its vocabulary is that of the English BERT. How does it cut "Immunoglobulin", and how a Turkish clinical sentence? Why does a model need far more training on Turkish text before the pieces of a Turkish word mean anything to it?

#### Fine-tuning the model for entities

*Lecture page: [Fine-tuning a pretrained model for entities](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#fine-tuning-a-pretrained-model-for-entities)*

The pretrained model gets a layer that gives every token one of seven tags, and the whole model is trained on the annotated training summaries for four epochs. The entities that it finds are judged by exact match on the test set and on hospital B.

**From the formulas to the code.** The tags of the BIO scheme: `TAGS`. A token gets the tag of the entity in which its first character lies: `e["start"] <= start < e["end"]` in `encode`. The loss is the cross-entropy over the tokens with a tag; the special tokens and the padding get −100, which the loss leaves out. Precision, recall and F1 of the entities: `span_scores`.

**Start cell.** For the section «Fine-tuning the model for entities», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Fine-tuning the model for entities»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
archive = reports.archive()
texts = archive["text_tr"].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

**On the lecture page:** [Worked example: An entity that is almost right](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-an-entity-that-is-almost-right). The cells below judge the entities of the model by exact match in the same way.

In [ ]:
TAGS = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
MAX_LENGTH = 384


def encode(text, entities):
    """Tokens of a report with one tag each: B- for the first token of an entity, I- for the others, O outside; -100 for the special tokens."""
    enc = tokenizer(tr_lower(text), return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
    tags = []
    for start, end in enc["offset_mapping"]:
        if start == end:
            tags.append(-100)
            continue
        tag = "O"
        for e in entities:
            if e["start"] <= start < e["end"]:
                tag = ("B-" if start == e["start"] else "I-") + e["label"]
                break
        tags.append(TAGS.index(tag))
    return enc["input_ids"], tags, enc["offset_mapping"]


def spans(tag_ids, offsets):
    """Entities from the tags of the tokens: (start, end, type) in characters of the report."""
    found, current = [], None
    for t, (start, end) in zip(tag_ids, offsets):
        if start == end:
            continue
        tag = TAGS[t]
        if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
            if current:
                found.append(tuple(current))
            current = [start, end, tag[2:]]
        elif tag.startswith("I-"):
            current[1] = end
        else:
            if current:
                found.append(tuple(current))
            current = None
    if current:
        found.append(tuple(current))
    return found


def batches(items, size, shuffle=False):
    """Groups of reports padded to the same length, for training and prediction."""
    order = np.random.permutation(len(items)) if shuffle else np.arange(len(items))
    for s in range(0, len(items), size):
        group = [items[i] for i in order[s:s + size]]
        width = max(len(g[0]) for g in group)
        ids = torch.tensor([g[0] + [0] * (width - len(g[0])) for g in group])
        tags = torch.tensor([g[1] + [-100] * (width - len(g[1])) for g in group])
        mask = torch.tensor([[1] * len(g[0]) + [0] * (width - len(g[0])) for g in group])
        yield ids, tags, mask


def span_scores(gold, predicted, kind=None):
    """Exact-match precision, recall and F1 of entities; kind limits them to one type."""
    tp = fp = fn = 0
    for g, p in zip(gold, predicted):
        g = {(e[0], e[1], e[2]) for e in g if kind is None or e[2] == kind}
        p = {e for e in p if kind is None or e[2] == kind}
        tp, fp, fn = tp + len(g & p), fp + len(p - g), fn + len(g - p)
    precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
    return round(precision, 3), round(recall, 3), round(2 * tp / max(1, 2 * tp + fp + fn), 3)

ids, tags, offsets = encode(texts[case], archive["entities_tr"][case])
for t, tag, (s, e) in list(zip(tokenizer.convert_ids_to_tokens(ids), tags, offsets))[30:48]:
    print(f"{t:14} {TAGS[tag] if tag >= 0 else '':8} {texts[case][s:e]}")

**Python note.** `encode` calls the tokenizer with `return_offsets_mapping=True`, which returns for every token the positions of its first and its last character in the text. Since `tr_lower` keeps the length of the text, these positions are those of the original summary. `spans` walks through the predicted tags and joins B- and the I- tags that follow it into one entity. `batches` pads the summaries of a group to the same length with the token 0 and builds `mask`, which tells the model which tokens are real.

In [ ]:
EPOCHS, BATCH, LEARNING_RATE, SEED = 4, 16, 3e-4, 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
train_items = [encode(t, e)[:2] for t, e in zip(texts[tr], archive["entities_tr"][tr])]
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS))
optimizer = torch.optim.AdamW(ner_model.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
    ner_model.train()
    total = 0.0
    for ids, tags, mask in batches(train_items, BATCH, shuffle=True):
        loss = ner_model(input_ids=ids, attention_mask=mask, labels=tags).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(ids)
    print(f"epoch {epoch + 1}: mean loss {total / len(train_items):.4f}")


def predict_spans(text_list):
    """Entities found by the model in every report of a list."""
    ner_model.eval()
    out = []
    with torch.no_grad():
        for text in text_list:
            ids, _, offsets = encode(text, [])
            logits = ner_model(input_ids=torch.tensor([ids])).logits[0]
            out.append(spans(logits.argmax(-1).tolist(), offsets))
    return out

results = {}
for name, mask in (("test", te), ("external", ex)):
    gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    predicted = predict_spans(list(texts[mask]))
    results[name] = predicted
    print(name, {kind or "all": span_scores(gold, predicted, kind) for kind in (None, "DIAG", "DRUG", "PROC")})

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c198)*

**Python note.** The training loop is that of Weeks 3 to 5, with three differences. The model comes ready from `from_pretrained`, with a new last layer of seven outputs. The batch is a dictionary of tensors that the model takes by name, `input_ids`, `attention_mask` and `labels`, and with `labels` the model computes the loss itself, `.loss`. The optimiser is AdamW, Adam with a weight decay that suits transformers. `torch.no_grad()` switches off the gradients during prediction. `torch.manual_seed(SEED)` and `np.random.seed(SEED)` fix the random start of the new layer and the order of the groups, so that a second run on the same computer gives the same result.

**Observe and reflect.** Print some entities of hospital B that the model missed or cut differently. Are they words that hospital A never writes, or entities whose edges differ by a suffix?

**Observe and reflect.** Set `SEED` to another number and train the model again. Which F1 changes more, that of the test set or that of hospital B? What does this say about a model that is tested only on the hospital it learned from?

**Exercise G.** How many tokens of the 1,800 training summaries get a beginning tag, B-DIAG, B-DRUG or B-PROC, with `encode`?

In [ ]:
data = exercise_table("G")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = 0
for t, es in zip(texts[tr], archive["entities_tr"][tr]):
    answer += sum(1 for x in encode(t, es)[1] if x >= 0 and TAGS[x].startswith("B-"))
teach.check("Exercise G", answer)

#### From entities to diagnoses

*Lecture page: [Evidence without a name: Drugs and laboratory values](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#evidence-without-a-name-drugs-and-laboratory-values)*

The entities of the model become the six diagnoses of every summary: a diagnosis entity is linked to one of the six by its words or, for a misspelling, by the most similar word of the dictionary; its assertion comes from the rule of «Stop 1 · Ready application»; drug and procedure entities and the two laboratory values add the rest.

**From the formulas to the code.** The similarity of two words is the ratio of `difflib.SequenceMatcher`, between 0 and 1, in `normalise`. The annotated entities give the upper limit of the pipeline, `entity_labels(text, gold)`; the entities of the model give its result.

**Start cell.** For the section «From entities to diagnoses», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «From entities to diagnoses»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower, se_ppv_f1, next_other, lab_value

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import difflib
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def label_table(y_true, y_pred):
    """One row per diagnosis: prevalence, sensitivity, PPV and F1; the last row is the mean F1 (macro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["diagnosis", "with the diagnosis", "sensitivity", "PPV", "F1"])
    print("Macro F1:", round(float(table["F1"].mean()), 3))
    return table
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Every mention of a diagnosis: (start, end, diagnosis), in the order of the text."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
def assertion(text, start, end, next_start, window=None):
    """The assertion of one mention: present, negated, possible or family.

    The scope after the mention ends at the end of the sentence or where the next mention of another diagnosis starts (next_start); with a window, it ends after that many characters instead."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
TAGS = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
MAX_LENGTH = 384
def encode(text, entities):
    """Tokens of a report with one tag each: B- for the first token of an entity, I- for the others, O outside; -100 for the special tokens."""
    enc = tokenizer(tr_lower(text), return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
    tags = []
    for start, end in enc["offset_mapping"]:
        if start == end:
            tags.append(-100)
            continue
        tag = "O"
        for e in entities:
            if e["start"] <= start < e["end"]:
                tag = ("B-" if start == e["start"] else "I-") + e["label"]
                break
        tags.append(TAGS.index(tag))
    return enc["input_ids"], tags, enc["offset_mapping"]
def spans(tag_ids, offsets):
    """Entities from the tags of the tokens: (start, end, type) in characters of the report."""
    found, current = [], None
    for t, (start, end) in zip(tag_ids, offsets):
        if start == end:
            continue
        tag = TAGS[t]
        if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
            if current:
                found.append(tuple(current))
            current = [start, end, tag[2:]]
        elif tag.startswith("I-"):
            current[1] = end
        else:
            if current:
                found.append(tuple(current))
            current = None
    if current:
        found.append(tuple(current))
    return found
def batches(items, size, shuffle=False):
    """Groups of reports padded to the same length, for training and prediction."""
    order = np.random.permutation(len(items)) if shuffle else np.arange(len(items))
    for s in range(0, len(items), size):
        group = [items[i] for i in order[s:s + size]]
        width = max(len(g[0]) for g in group)
        ids = torch.tensor([g[0] + [0] * (width - len(g[0])) for g in group])
        tags = torch.tensor([g[1] + [-100] * (width - len(g[1])) for g in group])
        mask = torch.tensor([[1] * len(g[0]) + [0] * (width - len(g[0])) for g in group])
        yield ids, tags, mask
def span_scores(gold, predicted, kind=None):
    """Exact-match precision, recall and F1 of entities; kind limits them to one type."""
    tp = fp = fn = 0
    for g, p in zip(gold, predicted):
        g = {(e[0], e[1], e[2]) for e in g if kind is None or e[2] == kind}
        p = {e for e in p if kind is None or e[2] == kind}
        tp, fp, fn = tp + len(g & p), fp + len(p - g), fn + len(g - p)
    precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
    return round(precision, 3), round(recall, 3), round(2 * tp / max(1, 2 * tp + fp + fn), 3)
EPOCHS, BATCH, LEARNING_RATE, SEED = 4, 16, 3e-4, 2026
np.random.seed(SEED)
train_items = [encode(t, e)[:2] for t, e in zip(texts[tr], archive["entities_tr"][tr])]
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS))
optimizer = torch.optim.AdamW(ner_model.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
    ner_model.train()
    total = 0.0
    for ids, tags, mask in batches(train_items, BATCH, shuffle=True):
        loss = ner_model(input_ids=ids, attention_mask=mask, labels=tags).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(ids)
    print(f"epoch {epoch + 1}: mean loss {total / len(train_items):.4f}")
def predict_spans(text_list):
    """Entities found by the model in every report of a list."""
    ner_model.eval()
    out = []
    with torch.no_grad():
        for text in text_list:
            ids, _, offsets = encode(text, [])
            logits = ner_model(input_ids=torch.tensor([ids])).logits[0]
            out.append(spans(logits.argmax(-1).tolist(), offsets))
    return out
results = {}
for name, mask in (("test", te), ("external", ex)):
    gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    predicted = predict_spans(list(texts[mask]))
    results[name] = predicted
    print(name, {kind or "all": span_scores(gold, predicted, kind) for kind in (None, "DIAG", "DRUG", "PROC")})

In [ ]:
ALL_TERMS = [(t, label) for label, terms in TERMS.items() for t in terms]
DRUG_EVIDENCE = {w: label for label, ws in EVIDENCE.items() for w in ws}


def normalise(mention):
    """The diagnosis of a mention: the diagnosis of a dictionary word that it contains, otherwise that of the most similar dictionary word, when the similarity is at least 0.8."""
    found = find_terms(mention)
    if found:
        return found[0][2]
    low = tr_lower(mention)
    score, label = max((difflib.SequenceMatcher(None, low, t).ratio(), lab) for t, lab in ALL_TERMS)
    return label if score >= 0.8 else None


def entity_labels(text, entities):
    """The six diagnoses of a report from its entities, each (start, end, type): diagnoses with their assertion, drugs and procedures, and two laboratory values."""
    mentions = sorted((s, e, normalise(text[s:e])) for s, e, kind in entities if kind == "DIAG")
    mentions = [m for m in mentions if m[2]]
    present = set()
    for i, (s, e, label) in enumerate(mentions):
        if assertion(text, s, e, next_other(mentions, i, text)) == "present":
            present.add(label)
    for s, e, kind in entities:
        if kind in ("DRUG", "PROC"):
            low = tr_lower(text[s:e])
            present.update(label for w, label in DRUG_EVIDENCE.items() if re.search(r"(?<!\w)" + re.escape(w), low))
    egfr, hba1c = lab_value(text, "egfr"), lab_value(text, "hba1c")
    if egfr is not None and egfr < 60:
        present.add("CKD")
    if hba1c is not None and hba1c >= 6.5:
        present.add("DM")
    return [int(label in present) for label in LABELS]

for word in ["hipertansyon", "koroner arter hastaligi", "diyebet", "kalp yetmezligi", "nefes darlığı"]:
    print(f"{word:26} → {normalise(word)}")
for name, mask, key in (("test", te, "test"), ("external", ex, "external")):
    gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    print(name, "annotated entities:")
    print(label_table(y[mask], np.array([entity_labels(t, g) for t, g in zip(texts[mask], gold)])))
    print(name, "entities of the model:")
    print(label_table(y[mask], np.array([entity_labels(t, p) for t, p in zip(texts[mask], results[key])])))

**Python note.** `difflib.SequenceMatcher(None, a, b).ratio()` compares two strings letter by letter and gives 1 for the same text. `max(...)` over pairs (score, diagnosis) returns the pair with the highest score, and the diagnosis is kept only when the score reaches 0.8, so that "nefes darlığı" links to no diagnosis. `entity_labels` uses the positions of the entities instead of searching the text, so the same function works with the annotations and with the predictions.

**Observe and reflect.** Compare the two tables of hospital B. Which diagnoses lose their positive predictive value with the entities of the model? Print the entities of the model in a few summaries of hospital B and look at their last characters.

**Exercise H.** What is the macro F1 on the summaries of hospital B when `entity_labels` uses the annotated entities? Keep it with three decimals; it is the upper limit of the pipeline.

In [ ]:
data = exercise_table("H")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][ex]]
P = np.array([entity_labels(t, g) for t, g in zip(texts[ex], gold)])
answer = round(float(np.mean([se_ppv_f1(y[ex, k], P[:, k])[2] for k in range(len(LABELS))])), 3)
teach.check("Exercise H", answer)

**Mini project: STEMI, AF and heart failure with an NER pipeline.** *Lecture page: [Mini project: Automatic extraction of diagnoses (STEMI, AF, HF) from synthetic Turkish discharge summaries: An NER pipeline](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#mini-project-automatic-extraction-of-diagnoses-stemi-af-hf-from-synthetic-turkish-discharge-summaries-an-ner-pipeline)*

The pipeline of the mini project reads three diagnoses from the entities of the fine-tuned model: AF and heart failure as in the cell above, and STEMI when an acute coronary syndrome is present and the summary reports ST elevation. The registry of the virtual clinic knows which patients had a STEMI, so the pipeline can be judged on both hospitals.

**From the formulas to the code.** STEMI: a present entity "akut koroner sendrom" or "AKS" and "st yükselmesi" in the Turkish lower case, in `stemi_af_hf`. The truth: `registry["stemi_territory"] != "none"`.

In [ ]:
from vclinic import make_cohort

ACS_WORDS = ("akut koroner sendrom", "aks")
ST_UP = re.compile(r"st\s+yükselmesi")


def stemi_af_hf(text, entities):
    """STEMI, AF and heart failure of a discharge summary from its entities: AF and heart failure as in entity_labels; STEMI when an acute coronary syndrome is present and the text reports ST elevation."""
    six = dict(zip(LABELS, entity_labels(text, entities)))
    mentions = sorted((s, e, normalise(text[s:e])) for s, e, kind in entities if kind == "DIAG")
    mentions = [m for m in mentions if m[2]]
    acs = any(tr_lower(text[s:e]).strip() in ACS_WORDS and assertion(text, s, e, next_other(mentions, i, text)) == "present"
              for i, (s, e, label) in enumerate(mentions))
    return [int(acs and ST_UP.search(tr_lower(text)) is not None), six["AF"], six["HF"]]


registry = make_cohort(len(archive)).set_index("patient_id")
stemi_true = (registry.loc[archive["patient_id"], "stemi_territory"] != "none").astype(int).to_numpy()
truth3 = np.column_stack([stemi_true, y[:, LABELS.index("AF")], y[:, LABELS.index("HF")]])

rows = []
for name, mask, key in (("test, hospital A", te, "test"), ("external, hospital B", ex, "external")):
    P = np.array([stemi_af_hf(t, p) for t, p in zip(texts[mask], results[key])])
    for k, d in enumerate(["STEMI", "AF", "HF"]):
        se, ppv, f1 = se_ppv_f1(truth3[mask, k], P[:, k])
        rows.append({"set": name, "diagnosis": d, "patients": int(truth3[mask, k].sum()),
                     "sensitivity": round(se, 3), "PPV": round(ppv, 3), "F1": round(f1, 3)})
mini_table = pd.DataFrame(rows)
print(mini_table.to_string(index=False))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c215)*

**Python note.** `make_cohort(len(archive))` rebuilds the registry of the 3,600 patients of the archive; `.set_index("patient_id")` and `.loc[archive["patient_id"]]` put its rows in the order of the summaries. `np.column_stack` joins three columns of 0 and 1 into the table of the truth. `any(...)` over a generator stops at the first present mention of an acute coronary syndrome.

**Observe and reflect.** The pipeline finds no STEMI in hospital B. Read the summary of a patient of hospital B who had a STEMI: What does the hospital write about the course and the ECG? Which sentence would a cardiologist add so that any pipeline could find the STEMI?

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#summaries-and-large-language-models-as-clinical-assistants) and continue reading at «Automatic summarisation of radiology reports». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Angiography reports: the degree of stenosis and the vessel

*Lecture page: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#stop-3)*

The angiography reports of the virtual clinic are read here as a registry would read them. The pipeline of the lecture finds the arteries, the segments and the degrees of stenosis of a report, keeps the largest stenosis of every territory and counts the diseased vessels. It is applied first to one report, then to the test reports of hospital A and finally to the reports of hospital B, which are written in capitals.

**At the end of this stop:** The number of diseased vessels of the report of patient VC-2914 (3), and the share of reports in which the number of diseased vessels is right, on the test set of hospital A (0.959) and on hospital B (1.0).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 3 · Python basics

*Lecture page: [Angiography reports: The degree of stenosis and the vessel](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#angiography-reports-the-degree-of-stenosis-and-the-vessel)*

The ready application of this stop cuts an angiography report into sentences, reads every degree of stenosis as a number or a range, keeps the largest value of every territory and puts it into the categories of the course. The examples below introduce these constructs on a short report: splitting at a pattern, the groups of a regular expression, a dictionary built in one line and conditions that follow each other.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 3 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower

# repeated from the earlier cells: the data and the code that the cells below use
import re

In [ ]:
# Python practice: a small example for learning the language
report = "Endikasyon: Göğüs ağrısı. LAD proksimalde %80 darlık. 1. diyagonalde %50 darlık. RCA normal."
for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", report):
    print(tr_lower(sentence).startswith(("endikasyon", "öykü")), "|", sentence)

**Python note.** `re.split` cuts a text at every match of a pattern. `(?<=\.)` asks for a full stop before the place of the cut and `(?=[A-ZÇĞİÖŞÜ0-9])` for a capital letter or a digit after it, and neither is cut away. The report is therefore cut after every full stop that a space and a capital letter or a digit follow, and "1. diyagonalde" stays in one piece. `startswith` also takes a tuple of words and is true when the sentence starts with any of them: The ready application skips the sentences of the indication and of the history in this way.

In [ ]:
# Python practice: a small example for learning the language
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
sentence = "lad proksimalde %70-80, mid segmentte yüzde 40 darlık"
for m in re.finditer(NUMBER, sentence):
    print(m.group(0), "→", m.group(1), m.group(2), "→", int(m.group(2) or m.group(1)))

**Python note.** `re.finditer` returns every match of the pattern with its groups. The pattern finds a number after "%" or "yüzde", and `(?:\s?-\s?(\d{1,3}))?` an optional second number after a hyphen, so `m.group(2)` is `None` when the degree is a single number. `m.group(2) or m.group(1)` takes the second number when there is one and the first otherwise: A range counts as its upper end, as in the pipeline of the lecture page.

In [ ]:
# Python practice: a small example for learning the language
best = {t: 0 for t in ["LM", "LAD", "LCX", "RCA"]}
for territory, value in [("LAD", 70), ("RCA", 40), ("LAD", 90), ("LCX", 50)]:
    best[territory] = max(best[territory], value)
print(best)

**Python note.** `{t: 0 for t in [...]}` is a *dictionary comprehension*: It builds a dictionary with the value 0 for every territory. The loop then keeps the largest stenosis of every territory, since `max(best[territory], value)` replaces a value only by a larger one; the second stenosis of the LAD, 90 percent, replaces the first. The ready application fills the table of every report in this way, sentence by sentence.

In [ ]:
# Python practice: a small example for learning the language
def category(value):
    return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


print([category(v) for v in (30, 60, 75, 95, 100)])
print(sum(best[t] >= 70 for t in ("LAD", "LCX", "RCA")))

**Python note.** `a if condition else b` is a conditional expression, and several can follow each other: `category` reads its conditions from left to right and returns the value of the first one that holds, the categories of the lecture page from 0 below 50 percent to 4 for an occlusion. `best[t] >= 70` gives `True` or `False` for every territory, and `sum` counts every `True` as 1: here one diseased vessel, the LAD. The function `n_vessels` of the ready application counts the diseased vessels in the same way.

**Exercise 3.1.** A sentence of a report reads "cx orta segmentte %50-60, distalde %40 darlık". What is its largest degree of stenosis, in percent, when a range counts as its upper end?

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
answer = max(int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, "cx orta segmentte %50-60, distalde %40 darlık"))
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** The largest stenoses of a report are LM 40, LAD 70, LCX 95 and RCA 65 percent. How many diseased vessels does it have, counting the territories LAD, LCX and RCA with 70 percent or more?

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
table = {"LM": 40, "LAD": 70, "LCX": 95, "RCA": 65}
answer = sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
teach.check("Exercise 3.2", answer)

**«Stop 3 · Python basics» is complete.** Next comes «Stop 3 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#ai-pair-programming-part-6-reviewing-a-text-pipeline) and continue reading at «Working with an AI assistant, part 6: Reviewing a text pipeline».

### Stop 3 · Prompt and application

This part has three prompts: «One report, sentence by sentence», «All test reports of hospital A» and «The reports of hospital B, in capitals». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-06/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 3 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

from vclinic import reports
import re

if "angio" not in globals():
    archive_angio_ready = True
    def tr_lower(text):
        """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """The words of a text in lower case: runs of letters and digits."""
        return re.findall(r"\w+", tr_lower(text))

    angio = reports.angiography_archive()
    TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
    VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
               "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
               "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
    TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
    WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
             "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
    NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
    WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
    SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


    def stenoses(sentence):
        """Every degree of stenosis of a sentence in percent: numbers (a range counts as its upper end) and words."""
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
        values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
        return values


    def angio_table(text, coreference=True):
        """The largest stenosis of every territory in one report. A sentence ends at a full stop before a capital letter or a digit; a sentence without an artery belongs to the artery of the sentence before."""
        best = {t: 0 for t in TERRITORIES}
        last = None
        for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
            sentence = tr_lower(sentence)
            if sentence.startswith(SKIP):
                continue
            found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
            if found:
                last = found[0][1]
            elif not coreference:
                continue
            if last is None:
                continue
            for value in stenoses(sentence):
                best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
        return best


    def category(value):
        """0 below 50, 1 for 50 to 69, 2 for 70 to 89, 3 for 90 to 99 and 4 for an occlusion."""
        return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


    def n_vessels(table):
        """Number of diseased vessels: territories LAD, LCX and RCA with a stenosis of 70 percent or more."""
        return sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
_TER = ["LM", "LAD", "LCX", "RCA"]


def _reports(mask):
    """Angiography reports with the largest stenosis of every territory and the number of diseased vessels in the registry."""
    d = angio[mask].reset_index(drop=True)
    t = pd.DataFrame({"report": d["text_tr"]})
    for name, col in zip(_TER, ["max_LM", "max_LAD", "max_LCX", "max_RCA"]):
        t[name] = d[col]
    t["diseased vessels"] = d["n_vessels"]
    return t
print(len(angio), "angiography reports")

#### Prompt 3.1: One report, sentence by sentence

**Goal.** Read one angiography report as the pipeline of the lecture does.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants a program that reads a coronary angiography report written in Turkish and gives the largest stenosis of every coronary territory and the number of diseased vessels.  
> Data: The notebook holds a table named "data" with one row: "report" holds the coronary angiography report of patient VC-2914 of the virtual clinic, a synthetic registry, written in Turkish.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Show every sentence with the artery it belongs to and the values found in it.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the sentences, the largest stenosis of the four territories LM, LAD, LCX and RCA, and the number of diseased vessels, and keep the number of diseased vessels under the name "answer".  
> Test: The sentence "Proksimal segmentinde ileri derecede darlık saptandı." must belong to the RCA, and the number of the catheters, 6, must not appear as a stenosis.  

In [ ]:
# Data of Prompt 3.1: the table "data" that the prompt describes
data = pd.DataFrame({"report": angio.loc[angio["patient_id"] == "VC-2914", "text_tr"].tolist()})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.1
teach.check_step("Prompt 3.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"report": angio.loc[angio["patient_id"] == "VC-2914", "text_tr"].tolist()})
vessels = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
territory = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
words = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
            "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
skip = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def read(report, show=False, lower=turkish_lower):
    best = {"LM": 0, "LAD": 0, "LCX": 0, "RCA": 0}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", report):
        low = lower(sentence)
        if low.startswith(skip):
            continue
        found = sorted((m.start(), v) for v, p in vessels.items() for m in re.finditer(p, low))
        if found:
            last = found[0][1]
        if last is None:
            continue
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?", low)]
        values += [words[m.group(1)] for m in re.finditer(r"(?<!\w)(" + "|".join(sorted(words, key=len, reverse=True)) + r")(?!\w)", low)]
        for v in values:
            best[territory[last]] = max(best[territory[last]], v)
        if show:
            print(f"{last:4} {values!s:10} {sentence}")
    return best

table = read(data["report"].iloc[0], show=True)
answer = sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
print(table, "· diseased vessels:", answer)
teach.check_step("Prompt 3.1", "answer", globals())

**What the result shows.** The report describes 3-vessel disease: 95 percent in the proximal LAD, an occlusion of the circumflex and a severe stenosis of the proximal RCA, whose sentence names no artery and belongs to the RCA of the sentence before.

#### Prompt 3.2: All test reports of hospital A

**Goal.** Apply the pipeline to every test report and compare it with the registry.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how often a program that reads coronary angiography reports gives the same number of diseased vessels and the same degree of stenosis as the registry.  
> Data: The notebook holds a table named "data" with the 394 coronary angiography reports of the test set of hospital A of the virtual clinic, a synthetic registry, written in Turkish. Each row is a patient: "report" holds the report, the four columns "LM", "LAD", "LCX" and "RCA" give the largest stenosis in percent of the territories of the left main coronary artery, the left anterior descending artery with its diagonal branches, the left circumflex artery with its obtuse marginal branches, and the right coronary artery with the posterior descending artery, as recorded in the registry, and "diseased vessels" gives the number of the territories LAD, LCX and RCA with a stenosis of 70 percent or more.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Apply this to every report. Put every largest stenosis, of the program and of the registry, into one of five categories: below 50, 50 to 69, 70 to 89, 90 to 99 and 100 percent. Compute for each territory the share of reports in which the categories agree, and the share of reports in which the number of diseased vessels agrees. Show the reports in which the number of diseased vessels differs.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the four shares of the territories and the share for the number of diseased vessels, and keep the last, with three decimals, under the name "answer".  
> Test: Every share must lie between 0 and 1, and the program must find 3 diseased vessels in the report of patient VC-2914.  

In [ ]:
# Data of Prompt 3.2: the table "data" that the prompt describes
data = _reports(angio["split"].to_numpy() == "test")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.2
teach.check_step("Prompt 3.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.2 (run it only if the code of the assistant does not pass the check)
data = _reports(angio["split"].to_numpy() == "test")
vessels = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
territory = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
words = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
            "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
skip = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def read(report, show=False, lower=turkish_lower):
    best = {"LM": 0, "LAD": 0, "LCX": 0, "RCA": 0}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", report):
        low = lower(sentence)
        if low.startswith(skip):
            continue
        found = sorted((m.start(), v) for v, p in vessels.items() for m in re.finditer(p, low))
        if found:
            last = found[0][1]
        if last is None:
            continue
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?", low)]
        values += [words[m.group(1)] for m in re.finditer(r"(?<!\w)(" + "|".join(sorted(words, key=len, reverse=True)) + r")(?!\w)", low)]
        for v in values:
            best[territory[last]] = max(best[territory[last]], v)
        if show:
            print(f"{last:4} {values!s:10} {sentence}")
    return best

found = [read(r) for r in data["report"]]
category = lambda v: 0 if v < 50 else 1 if v < 70 else 2 if v < 90 else 3 if v < 100 else 4
for name, key in zip(["LM", "LAD", "LCX", "RCA"], ["LM", "LAD", "LCX", "RCA"]):
    agree = np.mean([category(f[key]) == category(v) for f, v in zip(found, data[name])])
    print(name, round(float(agree), 3))
n_found = np.array([sum(f[t] >= 70 for t in ("LAD", "LCX", "RCA")) for f in found])
answer = round(float(np.mean(n_found == data["diseased vessels"].to_numpy())), 3)
print("Number of diseased vessels right:", answer)
wrong = np.where(n_found != data["diseased vessels"].to_numpy())[0]
for i in wrong[:3]:
    print("—", data["report"].iloc[i][:300], "...")
teach.check_step("Prompt 3.2", "answer", globals())

**What the result shows.** The number of diseased vessels is right in 0.959 of the test reports. The reports where it is wrong name an artery without Turkish letters or with a spelling error, such as "Sirkumflex arter" or "Sag koroner arter": The pipeline does not recognise the artery and gives its stenosis to the artery of the sentence before.

#### Prompt 3.3: The reports of hospital B, in capitals

**Goal.** Apply the same pipeline to the reports of another hospital, written in capitals and in a short style.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether a program that reads the coronary angiography reports of one hospital also reads those of another hospital, written in capitals and in a short style.  
> Data: The notebook holds a table named "data" with the 373 coronary angiography reports of the hospital B of the virtual clinic, a synthetic registry, written in Turkish in capital letters. Each row is a patient: "report" holds the report, the four columns "LM", "LAD", "LCX" and "RCA" give the largest stenosis in percent of the territories of the left main coronary artery, the left anterior descending artery with its diagonal branches, the left circumflex artery with its obtuse marginal branches, and the right coronary artery with the posterior descending artery, as recorded in the registry, and "diseased vessels" gives the number of the territories LAD, LCX and RCA with a stenosis of 70 percent or more.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Apply this to every report and compute the share of reports in which the number of diseased vessels agrees with the registry. Then repeat it with the usual lower case of the programming language instead of the Turkish one and compute the share again.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two shares, and keep the share with the Turkish lower case, with three decimals, under the name "answer".  
> Test: The share with the Turkish lower case must be higher than with the usual lower case.  

In [ ]:
# Data of Prompt 3.3: the table "data" that the prompt describes
data = _reports(angio["split"].to_numpy() == "external")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.3
teach.check_step("Prompt 3.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.3 (run it only if the code of the assistant does not pass the check)
data = _reports(angio["split"].to_numpy() == "external")
vessels = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
territory = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
words = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
            "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
skip = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


def read(report, show=False, lower=turkish_lower):
    best = {"LM": 0, "LAD": 0, "LCX": 0, "RCA": 0}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", report):
        low = lower(sentence)
        if low.startswith(skip):
            continue
        found = sorted((m.start(), v) for v, p in vessels.items() for m in re.finditer(p, low))
        if found:
            last = found[0][1]
        if last is None:
            continue
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?", low)]
        values += [words[m.group(1)] for m in re.finditer(r"(?<!\w)(" + "|".join(sorted(words, key=len, reverse=True)) + r")(?!\w)", low)]
        for v in values:
            best[territory[last]] = max(best[territory[last]], v)
        if show:
            print(f"{last:4} {values!s:10} {sentence}")
    return best

truth = data["diseased vessels"].to_numpy()


def vessel_count(lower):
    return np.array([sum(f[t] >= 70 for t in ("LAD", "LCX", "RCA")) for f in (read(r, lower=lower) for r in data["report"])])


answer = round(float(np.mean(vessel_count(turkish_lower) == truth)), 3)
usual = round(float(np.mean(vessel_count(str.lower) == truth)), 3)
print("Usual lower case:", usual)
print("Turkish lower case:", answer)
teach.check_step("Prompt 3.3", "answer", globals())

**What the result shows.** With the Turkish lower case, the number of diseased vessels is right in 1.0 of the reports of hospital B, whose short style holds no misspelled artery. With the usual lower case, every İ of PROKSİMAL, DİSTAL and MİD becomes i with a dot, and the words that follow no longer match; so does every capital I, which becomes i instead of ı.

**«Stop 3 · Prompt and application» is complete.** Next comes «Stop 3 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#ai-pair-programming-part-6-reviewing-a-text-pipeline) and continue reading at «Working with an AI assistant, part 6: Reviewing a text pipeline».

### Stop 3 · Ready application

This part reads the angiography reports of both hospitals sentence by sentence, finds the artery and every degree of stenosis of each sentence, keeps the largest value of every territory and compares the categories and the number of diseased vessels with the registry.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 3 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re

#### Angiography reports: stenosis and vessel

*Lecture page: [Angiography reports: The degree of stenosis and the vessel](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#angiography-reports-the-degree-of-stenosis-and-the-vessel)*

The pipeline reads a report sentence by sentence, skips the sentences of the indication, the history, the ventriculography and the intervention, finds the artery of every sentence, reads every degree of stenosis as a number, a range or a word, and keeps the largest value of every territory.

**From the formulas to the code.** Territories: `TERRITORY`, the branches to their parent. Significant: 50 or more in the LM, 70 or more elsewhere. Categories: `category`. Number of diseased vessels: `n_vessels`, the territories LAD, LCX and RCA with 70 or more. A range is read as its upper end: `m.group(2) or m.group(1)` in `stenoses`.

In [ ]:
angio = reports.angiography_archive()
TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
         "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def stenoses(sentence):
    """Every degree of stenosis of a sentence in percent: numbers (a range counts as its upper end) and words."""
    values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
    values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
    return values


def angio_table(text, coreference=True):
    """The largest stenosis of every territory in one report. A sentence ends at a full stop before a capital letter or a digit; a sentence without an artery belongs to the artery of the sentence before."""
    best = {t: 0 for t in TERRITORIES}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
        sentence = tr_lower(sentence)
        if sentence.startswith(SKIP):
            continue
        found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
        if found:
            last = found[0][1]
        elif not coreference:
            continue
        if last is None:
            continue
        for value in stenoses(sentence):
            best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
    return best


def category(value):
    """0 below 50, 1 for 50 to 69, 2 for 70 to 89, 3 for 90 to 99 and 4 for an occlusion."""
    return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


def n_vessels(table):
    """Number of diseased vessels: territories LAD, LCX and RCA with a stenosis of 70 percent or more."""
    return sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))

angio_case = angio.set_index("patient_id").loc["VC-2914"]
print(angio_case["text_tr"])
print("Pipeline:", angio_table(angio_case["text_tr"]), "· registry:", {t: int(angio_case[f"max_{t}"]) for t in TERRITORIES})
for name, part in (("test", "test"), ("external", "external")):
    d = angio[angio["split"] == part]
    found = [angio_table(t) for t in d["text_tr"]]
    agree = {t: round(float(np.mean([category(f[t]) == c for f, c in zip(found, d[f"cat_{t}"])])), 3) for t in TERRITORIES}
    right = round(float(np.mean([n_vessels(f) == n for f, n in zip(found, d["n_vessels"])])), 3)
    print(name, agree, "· number of diseased vessels right:", right)

**Python note.** `re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text)` cuts the report after a full stop that a space and a capital letter or a digit follow, so that "1. diyagonal" stays in one piece. `sentence.startswith(SKIP)` takes a tuple and is true when the sentence starts with any of its words. `sorted(...)[0][1]` gives the artery that the sentence names first. `last` keeps the artery of the sentence before for the sentences that name none.

In [ ]:
d = angio[angio["split"] == "test"]
wrong = [(r["patient_id"], angio_table(r["text_tr"]), {t: int(r[f"max_{t}"]) for t in TERRITORIES}) for _, r in d.iterrows()
         if any(category(angio_table(r["text_tr"])[t]) != r[f"cat_{t}"] for t in TERRITORIES)]
print(len(wrong), "test reports with a wrong territory; the first:")
print(wrong[0])
print(angio.set_index("patient_id").loc[wrong[0][0], "text_tr"])

**Observe and reflect.** Read the first report that the pipeline gets wrong. Which sentence misled it, and what would you change in the pipeline? Would your change also work on reports that you have not seen?

**Exercise I.** What share of the test reports of hospital A get the right number of diseased vessels when the sentences without an artery are skipped, `angio_table(text, coreference=False)`? Keep it with three decimals.

In [ ]:
data = exercise_table("I")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d = angio[angio["split"] == "test"]
answer = round(float(np.mean([n_vessels(angio_table(t, coreference=False)) == n for t, n in zip(d["text_tr"], d["n_vessels"])])), 3)
teach.check("Exercise I", answer)

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#ai-pair-programming-part-6-reviewing-a-text-pipeline) and continue reading at «Working with an AI assistant, part 6: Reviewing a text pipeline». The weekly project is at the end of this Jupyter notebook, and the section «Weekly project» of the lecture page describes it.

## Summaries, trial eligibility and the check of an assistant

*Lecture page: [Automatic summarisation of radiology reports](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#summaries-and-large-language-models-as-clinical-assistants)*

Three short summaries of the case are given, written in the style of an AI assistant. The check of the lecture compares each with the discharge summary: every number must appear in the report, every drug must be a drug of the report, and every diagnosis that the summary calls present must be supported by the report, by a present mention, a drug or a laboratory value.

**From the formulas to the code.** Numbers of a text: `re.findall(r"\d+(?:,\d+)?", ...)`. The drugs of the archive: `DRUG_NAMES`, the first word of every drug entity. The diagnoses supported by the report: `rule_labels(report)`; those claimed by the summary: `assertion_labels(summary)`.

This part follows the practice stops and stands on its own: Its start cell repeats the setup and the data calls that its cells use.

**Start cell.** For «Summaries, trial eligibility and the check of an assistant», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» and «Stop 3 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Summaries, trial eligibility and the check of an assistant»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# functions of earlier parts, kept unchanged in the course package
from vclinic.week06 import tr_lower, se_ppv_f1, next_other, lab_value, n_vessels

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week06 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import re
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Every mention of a diagnosis: (start, end, diagnosis), in the order of the text."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
def assertion(text, start, end, next_start, window=None):
    """The assertion of one mention: present, negated, possible or family.

    The scope after the mention ends at the end of the sentence or where the next mention of another diagnosis starts (next_start); with a window, it ends after that many characters instead."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"
def assertion_labels(text, window=None):
    """A diagnosis counts when at least one of its mentions is present."""
    mentions = find_terms(text)
    present = set()
    for i, (start, end, label) in enumerate(mentions):
        if assertion(text, start, end, next_other(mentions, i, text), window) == "present":
            present.add(label)
    return [int(label in present) for label in LABELS]
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}
def rule_labels(text, window=None):
    """The mentions with their assertion, the drugs and two laboratory values."""
    labels = assertion_labels(text, window)
    low = tr_lower(text)
    for k, label in enumerate(LABELS):
        if any(re.search(r"(?<!\w)" + re.escape(w), low) for w in EVIDENCE.get(label, [])):
            labels[k] = 1
    egfr, hba1c = lab_value(text, "egfr"), lab_value(text, "hba1c")
    if egfr is not None and egfr < 60:
        labels[LABELS.index("CKD")] = 1
    if hba1c is not None and hba1c >= 6.5:
        labels[LABELS.index("DM")] = 1
    return labels
angio = reports.angiography_archive()
TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
         "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")
def stenoses(sentence):
    """Every degree of stenosis of a sentence in percent: numbers (a range counts as its upper end) and words."""
    values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
    values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
    return values
def angio_table(text, coreference=True):
    """The largest stenosis of every territory in one report. A sentence ends at a full stop before a capital letter or a digit; a sentence without an artery belongs to the artery of the sentence before."""
    best = {t: 0 for t in TERRITORIES}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
        sentence = tr_lower(sentence)
        if sentence.startswith(SKIP):
            continue
        found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
        if found:
            last = found[0][1]
        elif not coreference:
            continue
        if last is None:
            continue
        for value in stenoses(sentence):
            best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
    return best

In English, the three summaries say: (1) an 83-year-old woman with heart failure with preserved ejection fraction (EF 55%), eGFR 59, discharged on seven drugs of the report; (2) the same, with atrial fibrillation, on furosemide, spironolactone, apixaban and amlodipine; (3) heart failure with an EF of 35%, followed for coronary artery disease, eGFR 59, on furosemide and metoprolol.

**On the lecture page:** [Worked example: Checking a summary of the case](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#worked-example-checking-a-summary-of-the-case). The cell below checks the same summary and two more against the discharge summary of the case.

In [ ]:
summary_1 = "83 yaşında kadın hasta. Korunmuş ejeksiyon fraksiyonlu kalp yetersizliği (EF %55), eGFR 59. Taburcu tedavisi: furosemid, spironolakton, dapagliflozin, bisoprolol, perindopril, amlodipin, rosuvastatin."
summary_2 = "83 yaşında kadın hasta; korunmuş ejeksiyon fraksiyonlu kalp yetersizliği (EF %55) ve atriyal fibrilasyon; eGFR 59; furosemid, spironolakton, apiksaban ve amlodipin ile taburcu edildi."
summary_3 = "83 yaşında kadın hasta; kalp yetersizliği, EF %35; koroner arter hastalığı nedeniyle izleniyor; eGFR 59. Taburcu tedavisi: furosemid, metoprolol."

DRUG_NAMES = sorted({tr_lower(e["text"]).split()[0] for es in archive["entities_tr"] for e in es if e["label"] == "DRUG"}, key=len, reverse=True)


def check_summary(summary, report):
    """What a summary says that the report does not support: numbers, drugs and present diagnoses."""
    low_s, low_r = tr_lower(summary), tr_lower(report)
    problems = []
    numbers_r = set(re.findall(r"\d+(?:,\d+)?", low_r))
    problems += [f"number {n}" for n in re.findall(r"\d+(?:,\d+)?", low_s) if n not in numbers_r]
    for d in DRUG_NAMES:
        pattern = r"(?<!\w)" + re.escape(d) + r"(?!\w)"
        if re.search(pattern, low_s) and not re.search(pattern, low_r):
            problems.append(f"drug {d}")
    supported = rule_labels(report)
    claimed = assertion_labels(summary)
    problems += [f"diagnosis {LABELS[k]}" for k in range(len(LABELS)) if claimed[k] and not supported[k]]
    return problems

for name, summary in (("1", summary_1), ("2", summary_2), ("3", summary_3)):
    print(name, check_summary(summary, texts[case]) or "no problem found")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c234)*

**Python note.** `check_summary` collects its findings in the list `problems` with `+=` and `.append`, and returns it; an empty list counts as false, so `... or "no problem found"` prints the text instead. The numbers of the report are kept in a *set*, `set(...)`, in which `in` is fast. The diagnoses use the two functions of «Stop 1 · Ready application»: `rule_labels` for what the report supports, with its drugs and values, and `assertion_labels` for what the summary claims.

**Observe and reflect.** The check found the false atrial fibrillation and apixaban in summary 2. What does it not check? Think of "preserved ejection fraction", of a wrong date, and of a summary that leaves out a diagnosis.

**Exercise J.** How many problems does `check_summary` find in summary 3?

In [ ]:
data = exercise_table("J", summary_3=summary_3)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(check_summary(summary_3, texts[case]))
teach.check("Exercise J", answer)

**The conclusion of an imaging report.** The reports of imaging, from echocardiography to coronary angiography, end with a conclusion that summarises the findings. The cell below writes the conclusion of an angiography report from the table of the pipeline of «Stop 3 · Ready application», an extractive summary in which every number can be traced back to the report, and compares it with the conclusion that the values of the registry give.

In [ ]:
def conclusion(table):
    """The conclusion of an angiography report from the table of the pipeline: the number of diseased vessels and every significant stenosis."""
    n = n_vessels(table)
    significant = [f"{t} {v}%" if v < 100 else f"{t} occluded" for t, v in table.items() if v >= (50 if t == "LM" else 70)]
    if not significant:
        return "No significant stenosis."
    head = f"{n}-vessel disease" if n else ""
    if table["LM"] >= 50:
        head = "Left main disease" + (", " + head if head else "")
    return head + ": " + ", ".join(significant) + "."

for _, r in angio[angio["split"] == "test"].head(3).iterrows():
    print(r["patient_id"], "·", conclusion(angio_table(r["text_tr"])))
for name, part in (("test", "test"), ("external", "external")):
    d = angio[angio["split"] == part]
    same = np.mean([conclusion(angio_table(t)) == conclusion({k: int(r[f"max_{k}"]) for k in TERRITORIES})
                    for t, (_, r) in zip(d["text_tr"], d.iterrows())])
    print(name, "conclusions equal to those of the registry:", round(float(same), 3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c241)*

**Clinical trial eligibility.** *Lecture page: [Clinical trial eligibility detection](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#clinical-trial-eligibility-detection)*

A trial of the course, invented for this exercise, looks for patients with heart failure and an EF of 40 percent or less, aged 18 to 85 years. It excludes an eGFR below 30 mL/min/1.73 m² and an acute coronary syndrome. The cell reads the five criteria from every discharge summary, with the rules of this week, and compares the result with the registry, criterion by criterion and for the trial as a whole.

**From the formulas to the code.** The criteria of one summary: `screen(text)`, a dictionary of five yes-or-no values. Eligible: all inclusion criteria and no exclusion criterion, `eligible(c)`. The acute coronary syndrome must be present by the assertion rule, `acs_present`.

In [ ]:
from vclinic import make_cohort

registry = make_cohort(len(archive)).set_index("patient_id")
AGE = re.compile(r"(\d+)\s*(?:yaşında|y\b)")
EF = re.compile(r"(?:ejeksiyon fraksiyonu|ef)\s*%\s*(\d+)")
ACS_TEXT = re.compile(r"akut koroner sendrom|(?<!\w)aks(?!\w)")


def acs_present(text):
    """True when a mention of an acute coronary syndrome is present by the assertion rule, not negated."""
    return any(assertion(text, m.start(), m.end(), len(text)) == "present" for m in ACS_TEXT.finditer(tr_lower(text)))


def screen(text):
    """The five criteria of the trial of the course for one discharge summary: heart failure, EF of 40 percent or less, age 18 to 85 (inclusion); eGFR below 30, acute coronary syndrome (exclusion)."""
    low = tr_lower(text)
    age, ef, egfr = AGE.search(low), EF.search(low), lab_value(text, "egfr")
    return {"HF": dict(zip(LABELS, rule_labels(text)))["HF"] == 1,
            "EF≤40": ef is not None and int(ef.group(1)) <= 40,
            "18-85": age is not None and 18 <= int(age.group(1)) <= 85,
            "eGFR<30": egfr is not None and egfr < 30,
            "ACS": acs_present(text)}


def eligible(c):
    """Eligible: all inclusion criteria met and no exclusion criterion."""
    return c["HF"] and c["EF≤40"] and c["18-85"] and not c["eGFR<30"] and not c["ACS"]

reg = registry.loc[archive["patient_id"]]
truth_criteria = {"HF": y[:, LABELS.index("HF")] == 1, "EF≤40": reg["lvef"].to_numpy() <= 40, "18-85": reg["age"].between(18, 85).to_numpy(),
                  "eGFR<30": reg["egfr"].to_numpy() < 30, "ACS": reg["main_diagnosis"].to_numpy() == "ACS"}
truth_eligible = (truth_criteria["HF"] & truth_criteria["EF≤40"] & truth_criteria["18-85"] & ~truth_criteria["eGFR<30"] & ~truth_criteria["ACS"])
plain_acs = np.array([ACS_TEXT.search(tr_lower(t)) is not None for t in texts])
for name, mask in (("test, hospital A", te), ("external, hospital B", ex)):
    criteria = [screen(t) for t in texts[mask]]
    agree = {c: round(float(np.mean([x[c] for x in criteria] == truth_criteria[c][mask])), 3) for c in truth_criteria}
    found = np.array([eligible(x) for x in criteria])
    se, ppv, f1 = se_ppv_f1(truth_eligible[mask], found)
    print(name, "· agreement by criterion:", agree)
    print("   eligible in the registry:", int(truth_eligible[mask].sum()), "· found:", int(found.sum()),
          "· sensitivity", round(se, 3), "· PPV", round(ppv, 3),
          "· plain search for ACS wrong in:", int((plain_acs[mask] != truth_criteria["ACS"][mask]).sum()))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#nbref-c243)*

**Python note.** `screen` returns a dictionary, so `[x[c] for x in criteria]` takes one criterion from every summary. `reg["age"].between(18, 85)` is true for the ages from 18 to 85, the ends included. `~` turns True into False and back in an array, so `~truth_criteria["ACS"]` marks the patients without an acute coronary syndrome. `plain_acs` searches for the words alone, without the assertion rule.

**Observe and reflect.** A plain search for "akut koroner sendrom" would exclude many patients wrongly. Print one of these summaries. Why did the error change no decision of this trial, and in which trial would it lose eligible patients?

**«Summaries, trial eligibility and the check of an assistant» is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#summaries-and-large-language-models-as-clinical-assistants).

## Weekly project

*Lecture page: [Weekly project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#weekly-project) · [README of Week 6, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-06/README.md#weekly-project)*

**The project.** A pipeline for the coronary angiography reports of both hospitals: The largest stenosis of the four territories is read from every report, put into the five categories of the course and compared with the registry, together with left main disease, the number of diseased vessels, the reports that the pipeline reads wrongly and the discharge summaries of the same patients.

This part stands on its own: Its start cell loads the data of the project and prepares the tables of the steps. The project can be built with the prompts below, one for each step; each step works like a prompt of the stops, without a check cell or a ready code, because the results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

**Start cell.** For the weekly project, the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the weekly project
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="en", packages=["transformers"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

from vclinic import reports
import re

if "angio" not in globals():
    archive_angio_ready = True
    def tr_lower(text):
        """Lower case by the rules of Turkish: I becomes ı and İ becomes i, and the length of the text does not change."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """The words of a text in lower case: runs of letters and digits."""
        return re.findall(r"\w+", tr_lower(text))

    angio = reports.angiography_archive()
    TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
    VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
               "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
               "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
    TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
    WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
             "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
    NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
    WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
    SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


    def stenoses(sentence):
        """Every degree of stenosis of a sentence in percent: numbers (a range counts as its upper end) and words."""
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
        values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
        return values


    def angio_table(text, coreference=True):
        """The largest stenosis of every territory in one report. A sentence ends at a full stop before a capital letter or a digit; a sentence without an artery belongs to the artery of the sentence before."""
        best = {t: 0 for t in TERRITORIES}
        last = None
        for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
            sentence = tr_lower(sentence)
            if sentence.startswith(SKIP):
                continue
            found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
            if found:
                last = found[0][1]
            elif not coreference:
                continue
            if last is None:
                continue
            for value in stenoses(sentence):
                best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
        return best


    def category(value):
        """0 below 50, 1 for 50 to 69, 2 for 70 to 89, 3 for 90 to 99 and 4 for an occlusion."""
        return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


    def n_vessels(table):
        """Number of diseased vessels: territories LAD, LCX and RCA with a stenosis of 70 percent or more."""
        return sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
_TER = ["LM", "LAD", "LCX", "RCA"]


def _reports(mask):
    """Angiography reports with the largest stenosis of every territory and the number of diseased vessels in the registry."""
    d = angio[mask].reset_index(drop=True)
    t = pd.DataFrame({"report": d["text_tr"]})
    for name, col in zip(_TER, ["max_LM", "max_LAD", "max_LCX", "max_RCA"]):
        t[name] = d[col]
    t["diseased vessels"] = d["n_vessels"]
    return t
print(len(angio), "angiography reports")

if "archive" not in globals():
    archive = reports.archive()
    LABELS = list(reports.LABELS)
    texts = archive["text_tr"].to_numpy()
    y = archive[LABELS].to_numpy()
    split = archive["split"].to_numpy()
    tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


def _project_table():
    d = angio[angio["split"].isin(["test", "external"])].reset_index(drop=True)
    by_patient = archive.set_index("patient_id")["text_tr"]
    t = pd.DataFrame({"set": ["test" if s == "test" else "external" for s in d["split"]], "patient": d["patient_id"],
                      "report": d["text_tr"]})
    for name, col in zip(["LM", "LAD", "LCX", "RCA"], ["max_LM", "max_LAD", "max_LCX", "max_RCA"]):
        t[name] = d[col]
    t["diseased vessels"] = d["n_vessels"]
    t["summary"] = by_patient.loc[d["patient_id"]].to_numpy()
    return t

### Project step 1: Territory by territory in both hospitals

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to build a program that reads the coronary angiography reports of two hospitals for a registry, and to know how often it gives the same degree of stenosis as the registry in every coronary territory.  
> Data: The notebook holds a table named "data" with the 767 coronary angiography reports of the test set of hospital A (394) and of hospital B (373) of the virtual clinic, a synthetic registry, written in Turkish; hospital B writes in capitals and in a short style. Each row is a patient: "set" gives the set ("test" or "external"), "patient" the patient, "report" the report, the four columns "LM", "LAD", "LCX" and "RCA" the largest stenosis in percent of the four coronary territories as recorded in the registry, and "diseased vessels" the number of the territories LAD, LCX and RCA with a stenosis of 70 percent or more. "summary" holds the discharge summary of the same patient.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Apply this to every report. Put every largest stenosis, of the program and of the registry, into one of five categories: below 50, 50 to 69, 70 to 89, 90 to 99 and 100 percent. For each set and each territory, compute the share of reports in which the categories agree, and the share of reports in which all four territories agree.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each set and the five shares, and keep the share of reports of the set "test" in which all four territories agree, with three decimals, under the name "answer".  
> Test: There must be 394 reports in the set "test" and 373 in the set "external", and every share must lie between 0 and 1.  

In [ ]:
# Data of project step 1: the table "data" that the prompt describes
data = _project_table()
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: The left main coronary artery and the number of diseased vessels

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well a program that reads coronary angiography reports finds left main disease and the number of diseased vessels, the two findings that decide between surgery and stents.  
> Data: The notebook holds a table named "data" with the 767 coronary angiography reports of the test set of hospital A (394) and of hospital B (373) of the virtual clinic, a synthetic registry, written in Turkish; hospital B writes in capitals and in a short style. Each row is a patient: "set" gives the set ("test" or "external"), "patient" the patient, "report" the report, the four columns "LM", "LAD", "LCX" and "RCA" the largest stenosis in percent of the four coronary territories as recorded in the registry, and "diseased vessels" the number of the territories LAD, LCX and RCA with a stenosis of 70 percent or more. "summary" holds the discharge summary of the same patient.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Apply this to every report. Left main disease is a largest stenosis of 50 percent or more in the territory LM. For each set, compute the sensitivity and the positive predictive value of the program for left main disease against the column "LM", and show a table of the number of diseased vessels of the registry against that of the program.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two values for each set and the two tables, and keep the positive predictive value for left main disease in the set "test", with three decimals, under the name "answer".  
> Test: The numbers of the two tables must add up to the number of reports of the set.  

In [ ]:
# Data of project step 2: the table "data" that the prompt describes
data = _project_table()
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: The errors and their causes

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see every report that a program for coronary angiography reports reads wrongly, and to find the sentence that misled it.  
> Data: The notebook holds a table named "data" with the 767 coronary angiography reports of the test set of hospital A (394) and of hospital B (373) of the virtual clinic, a synthetic registry, written in Turkish; hospital B writes in capitals and in a short style. Each row is a patient: "set" gives the set ("test" or "external"), "patient" the patient, "report" the report, the four columns "LM", "LAD", "LCX" and "RCA" the largest stenosis in percent of the four coronary territories as recorded in the registry, and "diseased vessels" the number of the territories LAD, LCX and RCA with a stenosis of 70 percent or more. "summary" holds the discharge summary of the same patient.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Apply this to every report of the set "test". Put every largest stenosis, of the program and of the registry, into one of five categories: below 50, 50 to 69, 70 to 89, 90 to 99 and 100 percent. A report is wrong when at least one territory disagrees. For every wrong report, show the territories that disagree and the sentences that name no artery known to the program but contain a degree of stenosis.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the wrong reports with their territories and sentences, and keep the number of wrong reports under the name "answer".  
> Test: The number of wrong reports cannot be larger than 394.  

In [ ]:
# Data of project step 3: the table "data" that the prompt describes
data = _project_table()
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: The angiography and the discharge summary together

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how many patients with a significant coronary stenosis on angiography have no coronary artery disease in their discharge summary, as a search of discharge summaries would find it.  
> Data: The notebook holds a table named "data" with the 767 coronary angiography reports of the test set of hospital A (394) and of hospital B (373) of the virtual clinic, a synthetic registry, written in Turkish; hospital B writes in capitals and in a short style. Each row is a patient: "set" gives the set ("test" or "external"), "patient" the patient, "report" the report, the four columns "LM", "LAD", "LCX" and "RCA" the largest stenosis in percent of the four coronary territories as recorded in the registry, and "diseased vessels" the number of the territories LAD, LCX and RCA with a stenosis of 70 percent or more. "summary" holds the discharge summary of the same patient.  
> Task: Read the report sentence by sentence; a sentence ends at a full stop that is followed by a space and a capital letter or a digit. Bring every sentence to small letters by the Turkish rules (first I becomes ı and İ becomes i). Skip the sentences that start with "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" or "işlem". In every other sentence, find the first artery that it names: the left main "sol ana koroner" or "lmca"; the LAD "sol ön inen" or "lad", with its branches "birinci diyagonal", "1. diyagonal" or "d1" and "ikinci diyagonal", "2. diyagonal" or "d2"; the circumflex "sirkumfleks", "cx" or "lcx", with its branches "birinci obtus", "1. obtus" or "om1" and "ikinci obtus", "2. obtus" or "om2"; the RCA "sağ koroner" or "rca", with "arka inen", "posterior desendan" or "pda". Short names count only as whole words. A sentence without an artery belongs to the artery of the sentence before. Read every degree of stenosis of the sentence: a number after "%" or "yüzde " (a range such as "%70-80" counts as its upper end, 80), and the words "total oklüzyon", "tam tıkanıklık" or "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60), and "hafif", "plak" or "plaklı" (30). Give every value to the territory of its artery (the branches belong to the territory of their main artery) and keep the largest value of each territory; a territory without a value has 0. The number of diseased vessels counts the territories LAD, LCX and RCA with 70 or more. Apply this to every report; a patient has a significant stenosis when the program finds at least one diseased vessel or left main disease. Then read the discharge summary of the same patient: Bring every text to small letters by the Turkish rules: First turn I into ı and İ into i, then make the other letters small. Coronary artery disease is present in the summary when "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" or "akut koroner sendrom" occurs as a whole word and is not followed, before the end of its sentence or the next mention of another diagnosis, by "yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)", "şüphe", "olası" or "ön tanı", and is not preceded in its sentence by "babası", "annesi", "kardeşi", "babada", "annede" or "soygeçmiş", or when the summary names "klopidogrel", "tikagrelor", "pkg", "kabg" or "stent". For each set, count the patients with a significant stenosis whose summary has no coronary artery disease.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the counts for each set and three of these summaries, and keep the count of the set "test" under the name "answer".  
> Test: The count cannot be larger than the number of patients of the set with a significant stenosis.  

In [ ]:
# Data of project step 4: the table "data" that the prompt describes
data = _project_table()
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

**End of the weekly project.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lecture.html#weekly-project): The section «Weekly project» describes the report and its submission.

## Application challenges by field

| Field | Challenge |
|---|---|
| Heart failure | Add the subtype to the rule: heart failure with reduced ejection fraction when the summary gives an ejection fraction below 40 percent. How many patients of the test set does it find, and how would you check them against the registry? |
| Electrophysiology | The summaries of patients with atrial fibrillation list an anticoagulant. Count the patients with atrial fibrillation whose summary lists none, and write the half page that a quality committee would need about them. |
| Interventional cardiology | Extend the pipeline of «Stop 3 · Ready application» so that it also returns the segment of the largest stenosis of the LAD. How often is the proximal LAD diseased in the patients with two-vessel disease? |
| Prevention | Read the LDL of every summary as a number and compare it with the targets of the patients with coronary artery disease. How many patients with coronary artery disease leave hospital with an LDL above 70 mg/dL and without a statin? |
| Nephrology and cardiology | The rule calls chronic kidney disease from one eGFR. Which patients of the archive have an eGFR between 55 and 60, and how would a second value three months later change their label? |
| Clinical research | Build a table of all 3,600 patients with the six diagnoses of the rule and the number of diseased vessels of the angiography. How many patients with a significant stenosis have no coronary artery disease in their discharge summary, and what does this mean for a registry built only from summaries? |
| Medical informatics | Write three new sentences in the style of hospital B that the rule of the course reads wrongly, and change the rule so that it reads them right without breaking the results of «Stop 1 · Ready application». |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-06/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of this week extends «Stop 3 · Ready application»: a pipeline that extracts the degree of stenosis and the vessel from the angiography reports of both hospitals, with its errors explained.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>